In [ ]:
import sys, zipfile

wheel = '/home/root/jupyter_notebooks/pynqp2p_pkg/vendor_wheels/getmac-0.9.5-py2.py3-none-any.whl'
extract_dir = '/home/root/jupyter_notebooks/pynqp2p_pkg/vendor_wheels/extracted'
zipfile.ZipFile(wheel).extractall(extract_dir)

sys.path.insert(0, extract_dir)
sys.path.insert(0, '/home/root/jupyter_notebooks/pynqp2p_pkg')

import pynqp2p
print('pynqp2p OK from', pynqp2p.__file__)


# PYNQ 302 - Referee Match Client

This notebook is the **complete, real competition client**: vision detection
(all five approaches from `PYNQ 301`), the full referee wire protocol, and
real Genesis simulated-arm control, in one self-contained notebook. It talks
to the actual `gridmind-referee` Arena Agent over `pynqp2p`, using the exact
wire protocol documented in `student-competition-guide.md`.

`PYNQ 301 - Memory Game Grid Detection` still exists separately for tuning
and testing your vision model in isolation (a local human-vs-computer
practice game, no networking) -- use it to dial in `DETECTION_APPROACH`
before a real match, then come back here to actually play.

## Choosing a detection approach

Set `DETECTION_APPROACH` in Match Configuration below to one of:

- `'yolo_full_frame'` -- YOLO on the whole camera frame; set `BOARD_CORNERS` for an angled camera.
- `'yolo_grid_crops'` -- YOLO on each grid cell individually (cropped and stretched to 416x416). More robust when the background is cluttered.
- `'aruco_border'` -- Print markers 3-6 at the four physical corners of the arena (3=TL, 4=TR, 5=BR, 6=BL). They auto-calibrate the board every turn so `BOARD_CORNERS` is never needed. YOLO detects objects.
- `'aruco_per_card'` -- Print markers 0-29 on the front of each card. The marker ID encodes the cell directly, so no perspective math is needed. YOLO detects objects; ArUco identifies which cell the face-up card is in. **Default.**
- `'aruco_border_grid_crops'` -- Same corner auto-calibration as `aruco_border` (markers 3-6, `BOARD_CORNERS` never needed), but each grid cell is still cropped and detected individually like `yolo_grid_crops` instead of running YOLO on the whole frame. More robust when the background is cluttered, without ever hand-tuning corners.

All five feed the same `detect_position(pos)` used during a real match --
switching approaches doesn't require touching any other code.

## What's automatic vs. manual

There is no per-turn button to click. Connect, click **Start**, switch to
**Play Mode**, and the notebook plays the entire match on its own: choosing
which two positions to flip each turn (a known matching pair if your board
memory already has one, otherwise an unrevealed position), sending
`flip_both`, running your vision model on each `card_revealed`, comparing
your two cards, and submitting `report_result` -- all in a background
thread the moment the referee's messages arrive.

Two things are still under your control:

- **Wait Mode / Play Mode** -- a safety toggle. In Wait Mode the notebook stays connected and keeps
  the board memory up to date from every `card_revealed` broadcast (yours and the opponent's), but
  never acts on its own turn -- useful while you're still setting up or want a dry run. Play Mode lets
  it actually play.
- **Queue Hint** -- type an object name any time (ideally during the opponent's turn) and it's sent
  the instant your turn starts, before the automatic flip. If your turn has already begun by the time
  you queue it, it's too late for that turn -- queue it earlier next time.

**Pausing** stops the background thread entirely (no messages drained at all). It does **not** pause
the referee's 120-second turn timer -- if you pause mid-turn, you can still time out. Use it between
turns or if you need to stop and debug; use Wait Mode instead if you just want to hold off on
autonomous play while staying connected.

## Genesis simulated arm (optional, purely cosmetic)

If Genesis is configured for your arena, `game_start` includes a
`genesis_team_id` (`"team_red"`/`"team_blue"`) and `genesis_url`. This
notebook automatically joins Genesis's competition scene as that team and
animates your arm on every flip via `flip_card(row, col)` -- purely for
visual effect. Every Genesis call is best-effort: a failure never affects
your real match, which is decided entirely by `report_result` as always.


## 1. Setup

Run these cells from the same folder as the `PYNQ 301 - Object Detection` notebook if possible --
this notebook expects the same local model files:

- `tf_yolov3_voc.xmodel`
- `img/voc_classes.txt`


In [ ]:
from pathlib import Path
import os
import sys
import time
import json
import re
import base64
import subprocess
import requests
import threading
import colorsys
import random
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display, Image, clear_output
from pynq_dpu import DpuOverlay
import pynqp2p

BOOTCAMP_301_DIR = Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/PYNQ 301 - Object Detection')
NOTEBOOK_DIR = Path.cwd()

if not (NOTEBOOK_DIR / 'tf_yolov3_voc.xmodel').exists() and BOOTCAMP_301_DIR.exists():
    NOTEBOOK_DIR = BOOTCAMP_301_DIR

MODEL_PATH = NOTEBOOK_DIR / 'tf_yolov3_voc.xmodel'
CLASSES_PATH = NOTEBOOK_DIR / 'img' / 'voc_classes.txt'

# Pre-game riddle solving reuses the shared halo Strix LLM helper from the
# ai_llm bootcamp notebook (server IP, model names, and chat plumbing all
# live there) -- see PYNQ_301-Memory_Game_Grid_Detection-LLM.ipynb.
AI_HELPER_DIR = Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/ai_llm')
if str(AI_HELPER_DIR) not in sys.path:
    sys.path.append(str(AI_HELPER_DIR))
import bootcamp_ai

print(f'Using notebook assets from: {NOTEBOOK_DIR}')


In [ ]:
overlay = DpuOverlay('dpu.bit')
overlay.load_model(str(MODEL_PATH))


## 1a. Board Clock

KV260 boards have no RTC battery, so the system clock resets on every reboot
(often to 1970 or whenever the image was built) -- this must be re-set every
session before connecting. A wrong clock mostly matters for HTTPS (pip
installs, cert validation) and for making log/hint timestamps meaningful; the
referee wire protocol itself is plain HTTP and doesn't care.


In [ ]:
def set_board_time(date_string):
    """Sets the board's system clock via `date -s`. Jupyter on these boards
    already runs as root, so no sudo is needed."""
    result = subprocess.run(['date', '-s', date_string], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Failed to set date: {result.stderr.strip()}')
    now = subprocess.run(['date'], capture_output=True, text=True).stdout.strip()
    print(f'Board time set to: {now}')


def sync_board_time_from_http(url):
    """Best-effort: derive the current time from an HTTP server's Date
    header instead of typing it in by hand -- point this at the broker, the
    master, or any other host reachable from the board."""
    import urllib.request
    import email.utils

    request = urllib.request.Request(url, method='HEAD')
    with urllib.request.urlopen(request, timeout=5) as response:
        date_header = response.headers.get('Date')
    if not date_header:
        raise RuntimeError(f'{url} did not return a Date header.')
    parsed = email.utils.parsedate_to_datetime(date_header)
    set_board_time(parsed.strftime('%Y-%m-%d %H:%M:%S'))


board_time_text = widgets.Text(value='', placeholder='YYYY-MM-DD HH:MM:SS', description='Set time:')
set_time_button = widgets.Button(description='Set Board Time', button_style='warning')
sync_time_url_text = widgets.Text(value='', placeholder='e.g. http://192.168.1.100:5000', description='Sync from:')
sync_time_button = widgets.Button(description='Sync From HTTP', button_style='info')
time_output = widgets.Output()


def on_set_time_clicked(_button):
    with time_output:
        clear_output(wait=True)
        try:
            if not board_time_text.value.strip():
                raise ValueError('Enter a date/time first, e.g. 2026-07-16 09:00:00')
            set_board_time(board_time_text.value.strip())
        except Exception:
            import traceback
            traceback.print_exc()


def on_sync_time_clicked(_button):
    with time_output:
        clear_output(wait=True)
        try:
            if not sync_time_url_text.value.strip():
                raise ValueError('Enter a reachable URL first, e.g. the broker address.')
            sync_board_time_from_http(sync_time_url_text.value.strip())
        except Exception:
            import traceback
            traceback.print_exc()


set_time_button.on_click(on_set_time_clicked)
sync_time_button.on_click(on_sync_time_clicked)

display(widgets.VBox([
    widgets.HTML('<h3>Board Clock</h3>'),
    widgets.HTML('<small>Set the board\'s clock manually, or sync it from a reachable server\'s HTTP Date header.</small>'),
    widgets.HBox([board_time_text, set_time_button]),
    widgets.HBox([sync_time_url_text, sync_time_button]),
    time_output,
]))


## 2. Match Configuration

Edit these before every match. `GRID_ROWS`/`GRID_COLS` and `REFEREE_ID` are given to you by the RC
team when your match is set up -- they must match the physical board and the arena you were assigned.
`MASTER_ID` and `TEAM_SECRET` are given once at registration and don't change between rounds --
`TEAM_SECRET` is optional (see section 8 below).

Board positions are named `<row letter><col number>`, e.g. `B3`, matching the referee's wire protocol
exactly (see `student-api-reference.html`).


In [ ]:
SERVER = '192.168.1.100:5000'      # RC team gives you this
BROKER_KEY = 'bootcamp2024'        # RC team gives you this
REFEREE_ID = 'arena-1-referee'     # RC team gives you this -- may change between rounds
MASTER_ID = 'master-referee'       # RC team gives you this -- does NOT change between rounds
TEAM_NAME = 'alpha'                # your team name, exactly as registered

# Shown to your team once at registration. Leave blank to skip self-reporting your
# MAC -- the operator can still enter it for you manually, this just saves a step.
TEAM_SECRET = ''

# Leave blank to use this board's real MAC address (pynqp2p.get_id()).
# Only override this for lab testing on a machine with no eth0.
BOARD_ID_OVERRIDE = ''

GRID_ROWS = 5   # lettered A..E
GRID_COLS = 6   # numbered 1..6

# Example for an angled camera -- top-left, top-right, bottom-right, bottom-left:
# BOARD_CORNERS = np.float32([[80, 60], [1180, 80], [1220, 690], [60, 700]])
BOARD_CORNERS = None

# One of 'yolo_full_frame', 'yolo_grid_crops', 'aruco_border', 'aruco_per_card',
# 'aruco_border_grid_crops' --
# see the notebook title cell above for what each means. Also changeable live via
# the Detection dropdown in the widget GUI (Section 10).
DETECTION_APPROACH = 'aruco_per_card'

# Internal variables kept for backward compatibility with helper functions.
# Set automatically by set_detection_approach(); do not edit directly.
DETECTION_MODE = 'full_frame'
CARD_POSITION_MODE = 'aruco_id'


## 3. YOLO Detection Helpers

Same YOLO flow as `PYNQ 301 - Object Detection`. If your class labels don't match what the referee's
golden grid expects, matches will silently fail -- the referee compares your reported `cls` strings
against its own answer key, so they must be the exact VOC class names in `voc_classes.txt`.


In [ ]:
def get_class(classes_path):
    with open(classes_path) as f:
        return [c.strip() for c in f.readlines()]


class_names = get_class(CLASSES_PATH)

# Lower this if detections are consistently missed; raise it if you get false positives.
YOLO_SCORE_THRESHOLD = 0.2

anchor_list = [10, 13, 16, 30, 33, 23, 30, 61, 62, 45, 59, 119, 116, 90, 156, 198, 373, 326]
anchors = np.array([float(x) for x in anchor_list]).reshape(-1, 2)

num_classes = len(class_names)
hsv_tuples = [(1.0 * x / num_classes, 1.0, 1.0) for x in range(num_classes)]
colors = [tuple(int(c * 255) for c in colorsys.hsv_to_rgb(*hsv)) for hsv in hsv_tuples]
random.seed(0)
random.shuffle(colors)
random.seed(None)


In [ ]:
def letterbox_image(image, size):
    ih, iw, _ = image.shape
    w, h = size
    scale = min(w / iw, h / ih)
    nw, nh = int(iw * scale), int(ih * scale)

    image = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_LINEAR)
    new_image = np.ones((h, w, 3), np.uint8) * 128
    h_start, w_start = (h - nh) // 2, (w - nw) // 2
    new_image[h_start:h_start + nh, w_start:w_start + nw, :] = image
    return new_image


def pre_process(image, model_image_size):
    image = image[..., ::-1]
    boxed_image = letterbox_image(image, tuple(reversed(model_image_size)))
    image_data = np.array(boxed_image, dtype='float32') / 255.0
    return np.expand_dims(image_data, 0)


def _get_feats(feats, anchors, num_classes, input_shape):
    num_anchors = len(anchors)
    anchors_tensor = np.reshape(np.array(anchors, dtype=np.float32), [1, 1, 1, num_anchors, 2])
    grid_size = np.shape(feats)[1:3]
    nu = num_classes + 5
    predictions = np.reshape(feats, [-1, grid_size[0], grid_size[1], num_anchors, nu])
    grid_y = np.tile(np.reshape(np.arange(grid_size[0]), [-1, 1, 1, 1]), [1, grid_size[1], 1, 1])
    grid_x = np.tile(np.reshape(np.arange(grid_size[1]), [1, -1, 1, 1]), [grid_size[0], 1, 1, 1])
    grid = np.array(np.concatenate([grid_x, grid_y], axis=-1), dtype=np.float32)

    box_xy = (1 / (1 + np.exp(-predictions[..., :2])) + grid) / np.array(grid_size[::-1], dtype=np.float32)
    box_wh = np.exp(predictions[..., 2:4]) * anchors_tensor / np.array(input_shape[::-1], dtype=np.float32)
    box_confidence = 1 / (1 + np.exp(-predictions[..., 4:5]))
    box_class_probs = 1 / (1 + np.exp(-predictions[..., 5:]))
    return box_xy, box_wh, box_confidence, box_class_probs


def correct_boxes(box_xy, box_wh, input_shape, image_shape):
    box_yx, box_hw = box_xy[..., ::-1], box_wh[..., ::-1]
    input_shape = np.array(input_shape, dtype=np.float32)
    image_shape = np.array(image_shape, dtype=np.float32)
    new_shape = np.around(image_shape * np.min(input_shape / image_shape))
    offset = (input_shape - new_shape) / 2.0 / input_shape
    scale = input_shape / new_shape
    box_yx = (box_yx - offset) * scale
    box_hw *= scale

    box_mins, box_maxes = box_yx - box_hw / 2.0, box_yx + box_hw / 2.0
    boxes = np.concatenate([box_mins[..., 0:1], box_mins[..., 1:2], box_maxes[..., 0:1], box_maxes[..., 1:2]], axis=-1)
    boxes *= np.concatenate([image_shape, image_shape], axis=-1)
    return boxes


def boxes_and_scores(feats, anchors, classes_num, input_shape, image_shape):
    box_xy, box_wh, box_confidence, box_class_probs = _get_feats(feats, anchors, classes_num, input_shape)
    boxes = np.reshape(correct_boxes(box_xy, box_wh, input_shape, image_shape), [-1, 4])
    box_scores = np.reshape(box_confidence * box_class_probs, [-1, classes_num])
    return boxes, box_scores


def nms_boxes(boxes, scores):
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = (x2 - x1 + 1) * (y2 - y1 + 1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        xx1, yy1 = np.maximum(x1[i], x1[order[1:]]), np.maximum(y1[i], y1[order[1:]])
        xx2, yy2 = np.minimum(x2[i], x2[order[1:]]), np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0.0, xx2 - xx1 + 1) * np.maximum(0.0, yy2 - yy1 + 1)
        ovr = inter / (areas[i] + areas[order[1:]] - inter)
        order = order[np.where(ovr <= 0.55)[0] + 1]
    return keep


def evaluate(yolo_outputs, image_shape, class_names, anchors, score_thresh=None):
    score_thresh = YOLO_SCORE_THRESHOLD if score_thresh is None else score_thresh
    anchor_mask = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]
    boxes, box_scores = [], []
    input_shape = np.array(np.shape(yolo_outputs[0])[1:3]) * 32

    for i in range(len(yolo_outputs)):
        b, s = boxes_and_scores(yolo_outputs[i], anchors[anchor_mask[i]], len(class_names), input_shape, image_shape)
        boxes.append(b)
        box_scores.append(s)

    boxes = np.concatenate(boxes, axis=0)
    box_scores = np.concatenate(box_scores, axis=0)
    mask = box_scores >= score_thresh
    boxes_, scores_, classes_ = [], [], []

    for c in range(len(class_names)):
        class_boxes = boxes[mask[:, c]]
        class_scores = box_scores[:, c][mask[:, c]]
        keep = nms_boxes(class_boxes, class_scores)
        boxes_.append(class_boxes[keep])
        scores_.append(class_scores[keep])
        classes_.append(np.ones(len(keep), dtype=np.int32) * c)

    if not boxes_:
        return np.array([]), np.array([]), np.array([])
    return np.concatenate(boxes_, axis=0), np.concatenate(scores_, axis=0), np.concatenate(classes_, axis=0)


In [ ]:
dpu = overlay.runner
inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()
shapeIn = tuple(inputTensors[0].dims)
shapeOut0, shapeOut1, shapeOut2 = (tuple(t.dims) for t in outputTensors)

input_data = [np.empty(shapeIn, dtype=np.float32, order='C')]
output_data = [
    np.empty(shapeOut0, dtype=np.float32, order='C'),
    np.empty(shapeOut1, dtype=np.float32, order='C'),
    np.empty(shapeOut2, dtype=np.float32, order='C'),
]
image_buffer = input_data[0]


def run(frame, score_thresh=None):
    image_size = frame.shape[:2]
    image_data = np.array(pre_process(frame, (416, 416)), dtype=np.float32)
    image_buffer[0, ...] = image_data.reshape(shapeIn[1:])

    job_id = dpu.execute_async(input_data, output_data)
    dpu.wait(job_id)

    yolo_outputs = [
        np.reshape(output_data[0], shapeOut0),
        np.reshape(output_data[1], shapeOut1),
        np.reshape(output_data[2], shapeOut2),
    ]
    return evaluate(yolo_outputs, image_size, class_names, anchors, score_thresh=score_thresh)


## 4. Grid Mapping

Positions are named `<row letter><col number>` (e.g. `B3`) to match the referee's wire protocol
exactly. If the camera is angled, set `BOARD_CORNERS` above to the four physical pixel corners
(top-left, top-right, bottom-right, bottom-left).


In [ ]:
def pos_name(row, col):
    return f'{chr(ord("A") + row)}{col + 1}'


def parse_pos(pos):
    row = ord(pos[0].upper()) - ord('A')
    col = int(pos[1:]) - 1
    return row, col


def board_source_corners(frame_shape):
    height, width = frame_shape[:2]
    if BOARD_CORNERS is not None:
        return np.float32(BOARD_CORNERS)
    return np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])


def board_transform(frame_shape):
    destination = np.float32([[0, 0], [GRID_COLS, 0], [GRID_COLS, GRID_ROWS], [0, GRID_ROWS]])
    return cv2.getPerspectiveTransform(board_source_corners(frame_shape), destination)


def frame_transform(frame_shape):
    source = np.float32([[0, 0], [GRID_COLS, 0], [GRID_COLS, GRID_ROWS], [0, GRID_ROWS]])
    return cv2.getPerspectiveTransform(source, board_source_corners(frame_shape))


def grid_cell_quad(row, col, frame_shape):
    to_frame = frame_transform(frame_shape)
    grid_points = np.float32([[[col, row], [col + 1, row], [col + 1, row + 1], [col, row + 1]]])
    return cv2.perspectiveTransform(grid_points, to_frame).reshape(4, 2).astype(np.float32)


def crop_grid_cell(frame, row, col):
    """Returns (crop, src_quad, dst_quad) -- the quads let a caller map a
    detection made in crop-space back to real frame coordinates (see
    crop_box_to_frame_box/crop_box_center_to_frame/crop_points_to_frame
    below), needed by the grid-crop and per-cell-ArUco detection paths."""
    width, height = GRID_CROP_SIZE
    src = grid_cell_quad(row, col, frame.shape)
    dst = np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])
    transform = cv2.getPerspectiveTransform(src, dst)
    crop = cv2.warpPerspective(frame, transform, (width, height))
    return crop, src, dst


def box_center(box):
    y_min, x_min, y_max, x_max = map(float, box)
    return (x_min + x_max) / 2.0, (y_min + y_max) / 2.0


def grid_position_for_point(point, frame_shape):
    point_arr = np.float32([[[float(point[0]), float(point[1])]]])
    grid_x, grid_y = cv2.perspectiveTransform(point_arr, board_transform(frame_shape))[0][0]
    col = int(np.clip(np.floor(grid_x), 0, GRID_COLS - 1))
    row = int(np.clip(np.floor(grid_y), 0, GRID_ROWS - 1))
    return {
        'row': row,
        'col': col,
        'cell': pos_name(row, col),
        'center': (float(point[0]), float(point[1])),
    }


def grid_position_for_aruco_id(marker_id, fallback_center):
    marker_id = int(marker_id)
    cell_index = marker_id % (GRID_ROWS * GRID_COLS)
    row = cell_index // GRID_COLS
    col = cell_index % GRID_COLS
    return {
        'row': row,
        'col': col,
        'cell': pos_name(row, col),
        'center': tuple(map(float, fallback_center)),
    }


def aruco_grid_position(marker_id, center, frame_shape):
    if CARD_POSITION_MODE == 'aruco_id':
        return grid_position_for_aruco_id(marker_id, center)
    return grid_position_for_point(center, frame_shape)


def crop_box_to_frame_box(crop_box, src_quad, dst_quad, frame_shape):
    y_min, x_min, y_max, x_max = map(float, crop_box)
    crop_corners = np.float32([[
        [x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max],
    ]])
    inverse_transform = cv2.getPerspectiveTransform(dst_quad, src_quad)
    frame_corners = cv2.perspectiveTransform(crop_corners, inverse_transform).reshape(4, 2)
    frame_height, frame_width = frame_shape[:2]
    xs = np.clip(frame_corners[:, 0], 0, frame_width - 1)
    ys = np.clip(frame_corners[:, 1], 0, frame_height - 1)
    return [float(np.min(ys)), float(np.min(xs)), float(np.max(ys)), float(np.max(xs))]


def crop_box_center_to_frame(crop_box, src_quad, dst_quad):
    y_min, x_min, y_max, x_max = map(float, crop_box)
    crop_center = np.float32([[[((x_min + x_max) / 2.0), ((y_min + y_max) / 2.0)]]])
    inverse_transform = cv2.getPerspectiveTransform(dst_quad, src_quad)
    center = cv2.perspectiveTransform(crop_center, inverse_transform)[0][0]
    return (float(center[0]), float(center[1]))


def crop_points_to_frame(points, src_quad, dst_quad):
    inverse_transform = cv2.getPerspectiveTransform(dst_quad, src_quad)
    frame_points = cv2.perspectiveTransform(np.float32([points]), inverse_transform)[0]
    return frame_points.astype(np.float32)


def draw_grid(frame):
    frame = frame.copy()
    to_frame = frame_transform(frame.shape)
    for col in range(GRID_COLS + 1):
        p1, p2 = cv2.perspectiveTransform(np.float32([[[col, 0]], [[col, GRID_ROWS]]]), to_frame).reshape(2, 2).astype(int)
        cv2.line(frame, tuple(p1), tuple(p2), (0, 255, 0), 1)
    for row in range(GRID_ROWS + 1):
        p1, p2 = cv2.perspectiveTransform(np.float32([[[0, row]], [[GRID_COLS, row]]]), to_frame).reshape(2, 2).astype(int)
        cv2.line(frame, tuple(p1), tuple(p2), (0, 255, 0), 1)
    return frame


def draw_label(frame, label, x, y, color=(255, 255, 255)):
    size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
    y = max(y, size[1] + 8)
    cv2.rectangle(frame, (x - size[0] // 2 - 4, y - size[1] - 8), (x + size[0] // 2 + 4, y + 4), (0, 0, 0), -1)
    return cv2.putText(frame, label, (x - size[0] // 2, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)


## 4a. ArUco Markers & Multi-Approach Detection Dispatch

Ported from `PYNQ 301 - Memory Game Grid Detection`'s detection engine, so
this notebook can use any of the four `DETECTION_APPROACH` values without
duplicating or reimplementing them. `records_for_current_mode(frame)` is
the single entry point every approach funnels through -- `detect_position`
below calls it, then picks out whichever record matches the requested
position.


In [ ]:
# Which ArUco dictionary the printed markers use, and which four IDs mark the
# arena's physical corners (aruco_border mode) vs. individual cards (aruco_per_card).
ARUCO_DICTIONARY_NAME = 'DICT_4X4_50'
BORDER_MARKER_TL = 3
BORDER_MARKER_TR = 4
BORDER_MARKER_BR = 6
BORDER_MARKER_BL = 5

# Fixed floor for crop-space detections -- distinct from the fallback ladder
# below, which only applies to detect_position's own retry loop.
GRID_CROP_SIZE = (416, 416)

# Lower thresholds detect_position retries at, in order, before giving up.
YOLO_FALLBACK_SCORE_THRESHOLDS = [0.15, 0.1, 0.05]

# How many fresh photos detect_position takes per threshold before moving to
# the next (lower) one -- a bit of multi-photo voting cushions against one
# unlucky frame (motion blur, a hand briefly in view) without adding much
# latency, since the fallback ladder only kicks in on genuine misses.
DETECT_PHOTO_COUNT = 2
DETECT_SETTLE_SECONDS = 0.3
DETECT_FLUSH_FRAMES = 8

# Only used by the debug overlay (show_detection_frame) -- how many of the
# top-scoring records to draw boxes/labels for.
MAX_OBJECTS = 2
TURN_DEBUG_SHOW_LABELS = True
TURN_DEBUG_SHOW_ALL_DETECTIONS = False

_APPROACH_SETTINGS = {
    'yolo_full_frame':        ('full_frame', 'yolo_center'),
    'yolo_grid_crops':        ('grid',       'yolo_center'),
    'aruco_border':           ('full_frame', 'yolo_center'),
    'aruco_per_card':         ('full_frame', 'aruco_id'),
    'aruco_border_grid_crops': ('grid',      'yolo_center'),
}


def set_detection_mode(mode):
    global DETECTION_MODE
    if mode not in ('grid', 'full_frame'):
        raise ValueError("mode must be 'grid' or 'full_frame'")
    DETECTION_MODE = mode
    return DETECTION_MODE


def set_card_position_mode(mode):
    global CARD_POSITION_MODE
    if mode not in ('yolo_center', 'aruco_id'):
        raise ValueError("mode must be 'yolo_center' or 'aruco_id'")
    CARD_POSITION_MODE = mode
    return CARD_POSITION_MODE


def set_detection_approach(approach):
    global DETECTION_APPROACH
    if approach not in _APPROACH_SETTINGS:
        raise ValueError(f"Unknown approach: {approach!r}. Choose from: {list(_APPROACH_SETTINGS)}")
    DETECTION_APPROACH = approach
    det_mode, pos_mode = _APPROACH_SETTINGS[approach]
    set_detection_mode(det_mode)
    set_card_position_mode(pos_mode)
    print(f'[detection] approach set to: {approach}')


def aruco_dictionary():
    if not hasattr(cv2, 'aruco'):
        raise RuntimeError('OpenCV ArUco support is not available. Install opencv-contrib-python or use an OpenCV build with cv2.aruco.')
    dictionary_id = getattr(cv2.aruco, ARUCO_DICTIONARY_NAME, None)
    if dictionary_id is None:
        raise ValueError(f'Unknown ArUco dictionary: {ARUCO_DICTIONARY_NAME}')
    return cv2.aruco.getPredefinedDictionary(dictionary_id)


def aruco_detector_parameters():
    if hasattr(cv2.aruco, 'DetectorParameters'):
        return cv2.aruco.DetectorParameters()
    return cv2.aruco.DetectorParameters_create()


def detect_aruco_corners_and_ids(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    dictionary = aruco_dictionary()
    parameters = aruco_detector_parameters()
    if hasattr(cv2.aruco, 'ArucoDetector'):
        detector = cv2.aruco.ArucoDetector(dictionary, parameters)
        corners, ids, rejected = detector.detectMarkers(gray)
    else:
        corners, ids, rejected = cv2.aruco.detectMarkers(gray, dictionary, parameters=parameters)
    if ids is None:
        return [], np.array([], dtype=np.int32)
    return corners, ids.flatten().astype(np.int32)


def _log_status(message):
    """Prints to the widget GUI's transparency log if it exists (contained,
    scrollable), or falls back to a plain print (e.g. in PYNQ 301, which has
    no widget GUI). Keeps high-frequency diagnostic messages -- like a
    flaky border marker warning that can fire on every detection attempt --
    from flooding the raw cell output and visually burying the widget
    layout during active play."""
    log_widget = globals().get('log_output')
    if log_widget is not None:
        with log_widget:
            print(message)
    else:
        print(message)


def calibrate_from_border_markers(frame):
    """Detect the 4 arena corner markers (BORDER_MARKER_TL/TR/BR/BL) and update BOARD_CORNERS.
    Returns the number found; BOARD_CORNERS is only updated once all 4 are
    visible. Called automatically by records_for_current_mode in aruco_border
    and aruco_border_grid_crops modes -- no need to call this yourself."""
    global BOARD_CORNERS
    corners, ids = detect_aruco_corners_and_ids(frame)
    corner_map = {BORDER_MARKER_TL: 0, BORDER_MARKER_TR: 1, BORDER_MARKER_BR: 2, BORDER_MARKER_BL: 3}
    found = {}
    for marker_corners, marker_id in zip(corners, ids):
        mid = int(marker_id)
        if mid in corner_map:
            found[corner_map[mid]] = np.mean(marker_corners.reshape(4, 2), axis=0)
    if len(found) == 4:
        BOARD_CORNERS = np.float32([found[0], found[1], found[2], found[3]])
    return len(found)


def aruco_records_for_frame(frame, detection_mode):
    """Detect ArUco markers on the full frame, mapping each to a grid cell.
    Border markers (BORDER_MARKER_TL/TR/BR/BL) are explicitly excluded so
    they never appear as false card detections -- checked by exact ID, not
    just a >= card_id_limit range, since the border IDs (3-6) now fall
    inside the card ID range (0..GRID_ROWS*GRID_COLS-1) instead of safely
    above it."""
    corners, ids = detect_aruco_corners_and_ids(frame)
    records = []
    border_marker_ids = {BORDER_MARKER_TL, BORDER_MARKER_TR, BORDER_MARKER_BR, BORDER_MARKER_BL}
    card_id_limit = GRID_ROWS * GRID_COLS
    for marker_corners, marker_id in zip(corners, ids):
        if int(marker_id) in border_marker_ids or int(marker_id) >= card_id_limit:
            continue
        points = marker_corners.reshape(4, 2).astype(np.float32)
        center = tuple(np.mean(points, axis=0).astype(float))
        grid_info = aruco_grid_position(marker_id, center, frame.shape)
        x_min, y_min = np.min(points, axis=0)
        x_max, y_max = np.max(points, axis=0)
        records.append({
            'description': f'aruco_{int(marker_id)}', 'score': 1.0, 'class_index': int(marker_id),
            'aruco_id': int(marker_id), 'row': grid_info['row'], 'col': grid_info['col'],
            'cell': grid_info['cell'], 'center': grid_info['center'],
            'corners': [[float(x), float(y)] for x, y in points],
            'box': [float(y_min), float(x_min), float(y_max), float(x_max)],
            'crop_box': [float(y_min), float(x_min), float(y_max), float(x_max)],
            'detection_mode': detection_mode, 'detector': 'aruco',
        })
    return sorted(records, key=lambda record: (record['row'], record['col'], record['aruco_id']))


def aruco_records_for_grid_cell(frame, row, col):
    """Crop one grid cell, detect ArUco markers in that stretched crop, and map them back to the frame."""
    crop, src_quad, dst_quad = crop_grid_cell(frame, row, col)
    corners, ids = detect_aruco_corners_and_ids(crop)
    records = []
    for marker_corners, marker_id in zip(corners, ids):
        crop_points = marker_corners.reshape(4, 2).astype(np.float32)
        frame_points = crop_points_to_frame(crop_points, src_quad, dst_quad)
        center = tuple(np.mean(frame_points, axis=0).astype(float))
        grid_info = aruco_grid_position(marker_id, center, frame.shape)
        crop_x_min, crop_y_min = np.min(crop_points, axis=0)
        crop_x_max, crop_y_max = np.max(crop_points, axis=0)
        frame_x_min, frame_y_min = np.min(frame_points, axis=0)
        frame_x_max, frame_y_max = np.max(frame_points, axis=0)
        records.append({
            'description': f'aruco_{int(marker_id)}', 'score': 1.0, 'class_index': int(marker_id),
            'aruco_id': int(marker_id), 'row': grid_info['row'], 'col': grid_info['col'],
            'cell': grid_info['cell'], 'center': grid_info['center'],
            'corners': [[float(x), float(y)] for x, y in frame_points],
            'box': [float(frame_y_min), float(frame_x_min), float(frame_y_max), float(frame_x_max)],
            'crop_box': [float(crop_y_min), float(crop_x_min), float(crop_y_max), float(crop_x_max)],
            'detection_mode': 'grid', 'detector': 'aruco',
        })
    return records


def yolo_record_from_box(frame_shape, box, score, class_index, detection_mode, crop_box=None):
    y_min, x_min, y_max, x_max = map(float, box)
    center = ((x_min + x_max) / 2.0, (y_min + y_max) / 2.0)
    grid_info = grid_position_for_point(center, frame_shape)
    return {
        'description': class_names[int(class_index)], 'score': float(score), 'class_index': int(class_index),
        'row': grid_info['row'], 'col': grid_info['col'], 'cell': grid_info['cell'], 'center': grid_info['center'],
        'box': [y_min, x_min, y_max, x_max],
        'crop_box': [float(value) for value in (crop_box if crop_box is not None else box)],
        'detection_mode': detection_mode, 'detector': 'yolo',
    }


def best_detection_for_grid_cell(frame, row, col, score_thresh=None):
    """Run YOLO on one perspective-corrected grid cell and map the best box back to the frame."""
    crop, src_quad, dst_quad = crop_grid_cell(frame, row, col)
    boxes, scores, classes = run(crop, score_thresh=score_thresh)
    if not scores.any():
        return None
    best_idx = int(np.argmax(scores))
    crop_box = boxes[best_idx]
    frame_box = crop_box_to_frame_box(crop_box, src_quad, dst_quad, frame.shape)
    record = yolo_record_from_box(frame.shape, frame_box, scores[best_idx], classes[best_idx], detection_mode='grid', crop_box=crop_box)
    record['row'] = int(row)
    record['col'] = int(col)
    record['cell'] = pos_name(row, col)
    record['center'] = crop_box_center_to_frame(crop_box, src_quad, dst_quad)
    return record


def yolo_records_for_grid(frame, score_thresh=None):
    records = []
    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            record = best_detection_for_grid_cell(frame, row, col, score_thresh=score_thresh)
            if record is not None:
                records.append(record)
    return sorted(records, key=lambda record: record['score'], reverse=True)


def aruco_records_for_grid(frame):
    records = []
    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            records.extend(aruco_records_for_grid_cell(frame, row, col))
    return sorted(records, key=lambda record: (record['row'], record['col'], record['aruco_id']))


def yolo_records_for_full_frame(frame, score_thresh=None):
    boxes, scores, classes = run(frame, score_thresh=score_thresh)
    if not scores.any():
        return []
    records = [
        yolo_record_from_box(frame.shape, box, score, class_index, detection_mode='full_frame')
        for box, score, class_index in zip(boxes, scores, classes)
    ]
    return sorted(records, key=lambda record: record['score'], reverse=True)


def record_selection_key(record):
    return (record.get('detector') == 'yolo', 'aruco_id' in record, record.get('score', 1.0))


def merge_aruco_with_yolo_records(yolo_records, aruco_records):
    """Attach nearest ArUco IDs to YOLO cards, optionally using marker IDs for stored positions."""
    merged_records = [dict(record) for record in yolo_records]
    used_aruco_indices = set()
    for yolo_record in merged_records:
        if not aruco_records:
            break
        yolo_center = np.array(yolo_record['center'], dtype=np.float32)
        candidate_indices = [i for i in range(len(aruco_records)) if i not in used_aruco_indices]
        if not candidate_indices:
            break
        best_index = min(
            candidate_indices,
            key=lambda index: np.linalg.norm(np.array(aruco_records[index]['center'], dtype=np.float32) - yolo_center),
        )
        aruco_record = aruco_records[best_index]
        aruco_center = tuple(map(float, aruco_record['center']))
        yolo_record['yolo_cell'] = yolo_record['cell']
        yolo_record['yolo_center'] = yolo_record['center']
        yolo_record['aruco_id'] = int(aruco_record['aruco_id'])
        yolo_record['aruco_score'] = float(aruco_record.get('score', 1.0))
        yolo_record['aruco_center'] = aruco_center
        yolo_record['corners'] = aruco_record['corners']
        if CARD_POSITION_MODE == 'aruco_id':
            grid_info = grid_position_for_aruco_id(aruco_record['aruco_id'], aruco_center)
            yolo_record['row'] = int(grid_info['row'])
            yolo_record['col'] = int(grid_info['col'])
            yolo_record['cell'] = grid_info['cell']
        used_aruco_indices.add(best_index)
    unmatched_aruco_records = [dict(r) for i, r in enumerate(aruco_records) if i not in used_aruco_indices]
    merged_records.extend(unmatched_aruco_records)
    return sorted(merged_records, key=record_selection_key, reverse=True)


def scan_grid_cells_for_objects(frame, score_thresh=None):
    """Crop/stretch each grid cell, then run YOLO and ArUco marker detection on each crop. ('yolo_grid_crops')"""
    yolo_records = yolo_records_for_grid(frame, score_thresh=score_thresh)
    aruco_records = aruco_records_for_grid(frame)
    return merge_aruco_with_yolo_records(yolo_records, aruco_records)


def full_frame_records(frame, score_thresh=None):
    """Run YOLO and ArUco marker detection on the full camera frame."""
    yolo_records = yolo_records_for_full_frame(frame, score_thresh=score_thresh)
    aruco_records = aruco_records_for_frame(frame, detection_mode='full_frame')
    return merge_aruco_with_yolo_records(yolo_records, aruco_records)


def records_for_current_mode(frame, score_thresh=None):
    """Detect face-up cards using the configured DETECTION_APPROACH -- the
    single entry point detect_position() calls below, regardless of which
    of the four approaches is active."""
    approach = globals().get('DETECTION_APPROACH', 'aruco_per_card')
    if approach == 'yolo_full_frame':
        return yolo_records_for_full_frame(frame, score_thresh=score_thresh)
    if approach == 'yolo_grid_crops':
        return scan_grid_cells_for_objects(frame, score_thresh=score_thresh)
    if approach == 'aruco_border':
        n = calibrate_from_border_markers(frame)
        if n < 4:
            _log_status(f'[aruco_border] {n}/4 border markers visible -- BOARD_CORNERS not updated')
        return yolo_records_for_full_frame(frame, score_thresh=score_thresh)
    if approach == 'aruco_border_grid_crops':
        # Same corner auto-calibration as aruco_border, but each grid cell
        # is still cropped and detected individually (scan_grid_cells_for_objects)
        # rather than running YOLO on the whole frame -- more robust when the
        # background is cluttered, same as yolo_grid_crops, without ever
        # needing BOARD_CORNERS set by hand.
        n = calibrate_from_border_markers(frame)
        if n < 4:
            _log_status(f'[aruco_border_grid_crops] {n}/4 border markers visible -- BOARD_CORNERS not updated')
        return scan_grid_cells_for_objects(frame, score_thresh=score_thresh)
    if approach == 'aruco_per_card':
        yolo_records = yolo_records_for_full_frame(frame, score_thresh=score_thresh)
        aruco_records = aruco_records_for_frame(frame, detection_mode='full_frame')
        return merge_aruco_with_yolo_records(yolo_records, aruco_records)
    raise ValueError(f"Unknown DETECTION_APPROACH: {approach!r}. Choose from: {list(_APPROACH_SETTINGS)}")


def turn_score_thresholds(base_threshold=None, fallback_thresholds=None):
    """Base threshold followed by lower fallback thresholds, descending, deduplicated."""
    base_threshold = YOLO_SCORE_THRESHOLD if base_threshold is None else base_threshold
    if fallback_thresholds is None:
        fallback_thresholds = YOLO_FALLBACK_SCORE_THRESHOLDS
    thresholds = [float(base_threshold)]
    for threshold in sorted({float(v) for v in fallback_thresholds}, reverse=True):
        if threshold < base_threshold and not any(np.isclose(threshold, existing) for existing in thresholds):
            thresholds.append(threshold)
    return thresholds


def draw_detection_label(frame, label, x, y, color=(255, 255, 255)):
    text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
    y = max(y, text_size[1] + 8)
    cv2.rectangle(frame, (x, y - text_size[1] - 8), (x + text_size[0] + 8, y + 4), (0, 0, 0), -1)
    return cv2.putText(frame, label, (x + 4, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)


def record_color(record):
    if not len(colors):
        return (255, 255, 255)
    return colors[int(record.get('class_index', 0)) % len(colors)]


def draw_aruco_marker(frame, record, color):
    points = np.array(record['corners'], dtype=np.int32)
    cv2.polylines(frame, [points], True, color, 2)
    marker_center = record.get('aruco_center', record['center'])
    center_x, center_y = map(int, marker_center)
    cv2.circle(frame, (center_x, center_y), 4, color, -1)
    return frame


def marker_label(record):
    if 'aruco_id' in record:
        if record.get('detector') == 'yolo':
            return f"{record['description']} {record['score']:.2f} ID {record['aruco_id']} {record['cell']}"
        return f"ID {record['aruco_id']} {record['cell']}"
    return f"{record['description']} {record['score']:.2f} {record['cell']}"


def annotate_records_frame(frame, records, turn_label='Current detection'):
    annotated = draw_grid(frame)
    sorted_records = sorted(records, key=lambda record: record.get('score', 1.0), reverse=True)
    debug_records = sorted_records if TURN_DEBUG_SHOW_ALL_DETECTIONS else sorted_records[:MAX_OBJECTS]
    for object_number, record in enumerate(debug_records, start=1):
        y_min, x_min, y_max, x_max = map(int, record['box'])
        color = record_color(record)
        if record.get('detector') == 'yolo':
            annotated = cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), color, 2)
        if 'corners' in record:
            annotated = draw_aruco_marker(annotated, record, color)
        elif record.get('detector') != 'yolo':
            annotated = cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), color, 2)
        if TURN_DEBUG_SHOW_LABELS:
            label = f"{object_number}: {marker_label(record)}"
            annotated = draw_detection_label(annotated, label, x_min, y_min, color)
    marker_count = len([r for r in records if 'aruco_id' in r])
    yolo_count = len([r for r in records if r.get('detector') == 'yolo'])
    mode_word = 'grid mode' if DETECTION_MODE == 'grid' else 'full-frame mode'
    status = f"{turn_label}: {mode_word}, found {yolo_count} YOLO object(s), {marker_count} ArUco marker(s)"
    annotated = draw_detection_label(annotated, status, 20, 30, (255, 255, 255))
    if len(records) < MAX_OBJECTS:
        warning = 'Expected two revealed cards. Check YOLO threshold, ArUco dictionary, grid alignment, lighting, or visibility.'
        annotated = draw_detection_label(annotated, warning, 20, 65, (0, 255, 255))
    return annotated


def show_detection_frame(frame, records, turn_label='Current detection'):
    annotated = annotate_records_frame(frame, records, turn_label=turn_label)
    _, encoded_frame = cv2.imencode('.jpeg', annotated)
    detection_display_handle.update(Image(data=encoded_frame.tobytes()))


def print_aruco_marker_report(records):
    marker_records = [r for r in records if 'aruco_id' in r]
    if not marker_records:
        print('No ArUco markers detected')
        return
    print('ArUco markers detected:')
    for record in sorted(marker_records, key=lambda item: (item['row'], item['col'], item['aruco_id'])):
        center_x, center_y = record.get('aruco_center', record['center'])
        print(f"  id={record['aruco_id']} cell={record['cell']} row={record['row'] + 1} col={record['col'] + 1} center=({center_x:.1f}, {center_y:.1f})")


## 5. Camera

Scans `/dev/video*` and opens the first camera that returns a frame.


In [ ]:
CAMERA_SIZE = (1920, 1080)
CAMERA_FPS = 15
CAMERA_BUFFER_SIZE = 1


def video_devices():
    devices = []
    for name in os.listdir('/dev'):
        if name.startswith('video') and name[5:].isdigit():
            devices.append((int(name[5:]), f'/dev/{name}'))
    return [device for _, device in sorted(devices)]


def open_camera(device):
    camera = cv2.VideoCapture(device, cv2.CAP_V4L2)
    camera.set(cv2.CAP_PROP_BUFFERSIZE, CAMERA_BUFFER_SIZE)
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_SIZE[0])
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_SIZE[1])
    camera.set(cv2.CAP_PROP_FPS, CAMERA_FPS)
    return camera


def scan_working_camera():
    for device in video_devices():
        camera = open_camera(device)
        ret, _ = camera.read()
        if camera.isOpened() and ret:
            print(f'Using camera from {device}')
            return camera, device
        camera.release()
    raise RuntimeError('Could not find a working /dev/video* camera.')


def camera_is_open():
    return 'cap' in globals() and cap is not None and cap.isOpened()


def reopen_camera():
    global cap, camera_device
    if camera_is_open():
        cap.release()
    cap, camera_device = scan_working_camera()
    return cap


def ensure_camera_open():
    if not camera_is_open():
        reopen_camera()
    return cap


def drain_camera_buffer(flush_frames=8):
    ensure_camera_open()
    for _ in range(max(0, flush_frames)):
        cap.grab()


def read_camera_frame(read_attempts=8):
    ensure_camera_open()
    for _ in range(max(1, read_attempts)):
        ret, frame = cap.read()
        if ret and frame is not None:
            return frame
        cap.grab()
        ret, frame = cap.retrieve()
        if ret and frame is not None:
            return frame
        time.sleep(0.05)
    return None


def capture_frame(flush_frames=8, settle_seconds=0.3, read_attempts=8):
    if settle_seconds > 0:
        time.sleep(settle_seconds)
    for attempt in range(2):
        drain_camera_buffer(flush_frames if attempt == 0 else 0)
        frame = read_camera_frame(read_attempts=read_attempts)
        if frame is not None:
            return frame
        reopen_camera()
    raise RuntimeError('Could not read a fresh frame from the camera.')


if 'cap' in globals() and cap is not None:
    cap.release()
try:
    cap, camera_device = scan_working_camera()
except RuntimeError as exc:
    print(f'{exc} Continuing without a camera -- see the Simulated Camera Override section below.')
    cap, camera_device = None, None
alignment_display_handle = display(None, display_id=True)
detection_display_handle = display(None, display_id=True)


def camera_alignment_snapshot():
    """Debug-only: one grid-overlay snapshot to position the camera and tune
    BOARD_CORNERS/GRID_ROWS/GRID_COLS. Call again for a fresh snapshot.
    Never called automatically -- this is setup tooling, not part of a match.

    Continuous loop-mode capture (repeatedly grabbing frames outside of a
    real card_revealed event) is disabled for now -- it competed with the
    match's own single-shot detect_position() calls for the same camera
    device on a separate thread, and could leave a board unresponsive if
    left running into an actual match. Re-run this cell manually instead."""
    frame = capture_frame(flush_frames=1, settle_seconds=0.0)
    annotated = draw_grid(frame)
    _, encoded = cv2.imencode('.jpeg', annotated)
    alignment_display_handle.update(Image(data=encoded.tobytes()))


## 6. Per-Position Detection

Called automatically by the match client whenever a `card_revealed` broadcast matches a position
you just flipped -- you never call this manually during a real match. `detect_position_debug()` is
provided for testing your vision setup before a match starts.


In [ ]:
def detect_position(pos, show=True):
    """Capture the board, detect every currently face-up card using the
    configured DETECTION_APPROACH (records_for_current_mode -- works
    identically across all four approaches), and return (class_name, score)
    for `pos` specifically. Retries a couple of fresh photos at
    progressively lower confidence thresholds if `pos` isn't found on the
    first attempt, mirroring PYNQ 301's per-turn voting."""
    for threshold in turn_score_thresholds():
        for photo in range(DETECT_PHOTO_COUNT):
            frame = capture_frame(
                flush_frames=DETECT_FLUSH_FRAMES if photo == 0 else 0,
                settle_seconds=DETECT_SETTLE_SECONDS if photo == 0 else 0.15,
            )
            records = records_for_current_mode(frame, score_thresh=threshold)
            match = next((r for r in records if r['cell'] == pos), None)
            if match is not None:
                if show:
                    show_detection_frame(frame, records, turn_label=f'{pos} detection')
                return match['description'], match.get('score', 1.0)

    print(f'[detect] WARNING: no object detected at {pos} even at the lowest threshold.')
    if show:
        frame = capture_frame(flush_frames=0, settle_seconds=0.0)
        show_detection_frame(frame, [], turn_label=f'{pos} detection')
    return 'unknown', 0.0


def detect_position_debug(pos):
    """Manual, one-off test of detection at a position. Does not touch
    match state or send anything to the referee -- setup/debug only."""
    cls, score = detect_position(pos, show=True)
    print(f'{pos}: {cls} (score={score:.2f})')
    return cls, score


def _annotate_full_frame_detections(frame, boxes, scores, classes):
    annotated = draw_grid(frame.copy())
    for box, score, class_idx in zip(boxes, scores, classes):
        y_min, x_min, y_max, x_max = map(int, box)
        color = colors[int(class_idx) % len(colors)]
        cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), color, 2)
        label = f'{class_names[int(class_idx)]} {score:.2f}'
        annotated = draw_label(annotated, label, (x_min + x_max) // 2, y_min, color)
    count_label = f'{len(boxes)} object(s) detected' if len(boxes) else 'No objects detected'
    return draw_label(annotated, count_label, frame.shape[1] // 2, 30, (255, 255, 255))


def camera_debug_one_shot():
    """Test mode: capture one frame, run real YOLO detection on the WHOLE
    frame (not cropped to a single position), and show every box/label found.
    Use this to sanity-check the camera and model before a match -- does not
    touch match state or send anything to the referee."""
    frame = capture_frame(flush_frames=4, settle_seconds=0.2)
    boxes, scores, classes = run(frame)
    annotated = _annotate_full_frame_detections(frame, boxes, scores, classes)
    _, encoded = cv2.imencode('.jpeg', annotated)
    detection_display_handle.update(Image(data=encoded.tobytes()))
    print(f'{len(boxes)} object(s) detected:')
    for score, class_idx in sorted(zip(scores, classes), reverse=True):
        print(f'  {class_names[int(class_idx)]}: {score:.2f}')
    return boxes, scores, classes



## 7a. Pre-Game Riddle Solving (local, via the halo Strix LLM)

The whole-board pre-game puzzle riddle is free-form text (e.g. "I speak
without a mouth and hear without ears...") -- unlike the row/column number
riddles used for hints, there's no fixed answer bank a lookup table could
solve. This section asks the shared halo Strix LLM to solve it, same
connection (`bootcamp_ai`) as `PYNQ_301-Memory_Game_Grid_Detection-LLM.ipynb`'s
riddle hint feature. Purely local -- nothing here talks to the referee,
judging who answers first is still done out loud to the human referee.


In [ ]:
PREGAME_RIDDLE_MODEL = 'Smart Helper'  # bootcamp_ai friendly model name
PREGAME_RIDDLE_TIMEOUT_SECONDS = 20


def build_pregame_riddle_prompt(riddle_text):
    return (
        'You are a fast riddle-solving assistant for a live competition. '
        'A player was just given this riddle and needs the answer immediately '
        'so they can say it out loud before their opponent:\n\n'
        f'"{riddle_text}"\n\n'
        'Reply with ONLY compact JSON, no extra words, in exactly this shape:\n'
        '{"answer": "<short answer, one or two words>"}'
    )


def _extract_json_object(reply_text):
    """Extract the first valid JSON object, including from fenced/prose replies."""
    text = str(reply_text or '')
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', text):
        try:
            value, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            return value
    raise ValueError(f'LLM reply did not contain valid JSON: {reply_text!r}')


def _fallback_answer_from_llm(reply_text):
    """Recover a short answer from common non-JSON LLM responses."""
    text = str(reply_text or '').strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.IGNORECASE).strip()
    match = re.search(r'["\']?answer["\']?\s*[:=]\s*["\']?([^"\'\n}\]]+)', text, re.IGNORECASE)
    if not match:
        match = re.search(r'\banswer\s+is\s+([^\n.!?]+)', text, re.IGNORECASE)
    if match:
        answer = match.group(1)
    else:
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        answer = lines[0] if lines else ''
    answer = re.sub(r'^(?:answer\s*[:=-]\s*)', '', answer, flags=re.IGNORECASE)
    answer = answer.strip().strip('`"\'{}[]').rstrip('.,;:').strip()
    if not answer:
        raise ValueError(f'LLM reply did not contain a usable answer: {reply_text!r}')
    return answer[:80].strip()


def _prompt_llm_with_timeout(prompt_text, timeout_seconds):
    """Run the helper in a daemon thread so its 300s HTTP timeout cannot hang the notebook."""
    result_queue = queue.Queue(maxsize=1)

    def worker():
        try:
            reply = bootcamp_ai.prompt(
                prompt_text,
                enforce_cooldown=False,
                session='pregame_riddle',
            )
            result_queue.put(('ok', reply))
        except Exception as exc:
            result_queue.put(('error', exc))

    threading.Thread(target=worker, daemon=True, name='pregame-riddle-llm').start()
    try:
        status, payload = result_queue.get(timeout=float(timeout_seconds))
    except queue.Empty as exc:
        raise TimeoutError(f'LLM did not respond within {timeout_seconds} seconds.') from exc
    if status == 'error':
        raise payload
    return payload


def solve_pregame_riddle_with_llm(riddle_text, model=None):
    """Return a best-guess answer without allowing a stalled or malformed LLM
    response to hang the match client."""
    riddle_text = str(riddle_text).strip()
    if not riddle_text:
        raise ValueError('Riddle text is empty.')
    bootcamp_ai.use_model(model or PREGAME_RIDDLE_MODEL)
    reply_text = _prompt_llm_with_timeout(
        build_pregame_riddle_prompt(riddle_text),
        PREGAME_RIDDLE_TIMEOUT_SECONDS,
    )
    used_fallback = False
    try:
        parsed = _extract_json_object(reply_text)
        answer = str(parsed.get('answer') or '').strip()
        if not answer:
            raise ValueError('JSON answer was empty.')
    except (TypeError, ValueError, json.JSONDecodeError):
        answer = _fallback_answer_from_llm(reply_text)
        used_fallback = True
    return {
        'riddle': riddle_text,
        'answer': answer,
        'raw_reply': reply_text,
        'used_fallback': used_fallback,
    }


## 8. Referee Connection

Thin wrapper around `pynqp2p` matching the wire protocol in `student-api-reference.html` exactly.
This class owns no game logic -- only send/poll plumbing -- so it can be tested independently of the
turn-taking state machine below.

Reveals both of a turn's positions in one atomic round-trip (`flip_both`) rather than the two
sequential single-position `flip` calls documented as the original protocol -- this is the faster of
the two supported protocols (see `PROJECT_STATE.md` for the measured comparison), which is why the
autonomous `MatchClient` below always uses it. The referee still accepts single `flip` calls too, for
any other client that wants them.

`join_competition(secret)` is a one-time, optional self-report of your board's MAC to the Master --
lets the operator's match-assign popup auto-fill it instead of typing it in by hand. Called
automatically on Connect if you filled in `TEAM_SECRET` above; harmless to skip, the operator can
always enter your MAC manually instead.


In [ ]:
class RefereeClient:
    def __init__(self, server, key, referee_id, team, master_id=None, board_id=None):
        pynqp2p.register(server, key)
        self.referee_id = referee_id
        self.master_id = master_id
        self.team = team
        self.board_id = board_id or pynqp2p.get_id()

    def send(self, message):
        pynqp2p.send(self.referee_id, json.dumps(message))

    def poll(self):
        """Drain and parse every queued message. A malformed line is logged
        and skipped rather than raised, since one bad line should never take
        down the match loop."""
        messages = []
        for raw in pynqp2p.receive_all():
            try:
                messages.append(json.loads(raw))
            except json.JSONDecodeError:
                print(f'[referee] skipping malformed message: {raw!r}')
        return messages

    def flip_both(self, pos1, pos2):
        self.send({'type': 'flip_both', 'team': self.team, 'pos1': pos1, 'pos2': pos2})

    def report_result(self, pos1, pos2, cls1, cls2, claim):
        self.send({
            'type': 'report_result', 'team': self.team,
            'pos1': pos1, 'pos2': pos2, 'cls1': cls1, 'cls2': cls2, 'claim': claim,
        })

    def request_hint(self, obj):
        self.send({'type': 'hint_request', 'team': self.team, 'object': obj})

    def join_competition(self, secret):
        """Self-reports this board's MAC to the Master's lobby, once, right
        after connecting -- lets the operator's match-assign popup auto-fill
        your MAC instead of typing it in by hand. Requires `master_id` (does
        not change between rounds, unlike `referee_id`). Entirely optional:
        skip it and the operator can still enter your MAC manually."""
        if not self.master_id:
            print('[referee] join_competition skipped: no MASTER_ID set.')
            return
        lobby_id = f'{self.master_id}-lobby'
        pynqp2p.send(lobby_id, json.dumps({
            'type': 'join', 'team': self.team, 'mac': self.board_id, 'secret': secret,
        }))


## 9. Match Client

Owns the turn-taking state machine, a background polling thread, and the autonomous play strategy.
The only externally-triggered actions are `set_play_mode(enabled)` and `queue_hint(obj)` -- everything
else (choosing positions, flipping, running your own detection, comparing your two cards, submitting
`report_result`, tracking turns) is driven automatically the moment referee messages arrive.

**Auto-flip strategy** (`_choose_pair`): if the board memory already has two unmatched positions with
the same detected class, flip that guaranteed pair. Otherwise flip an unrevealed position paired with
either another known position or a second unrevealed one -- the same shape of logic as the computer
opponent in `PYNQ 301`, just driven by real detections instead of a golden answer key. Always uses
`flip_both` (one round-trip) rather than two sequential `flip` calls, since that's the faster protocol
we measured (see `PROJECT_STATE.md`).


In [ ]:
class Stage:
    """Explicit lifecycle stages, replacing free-form status prose with a
    fixed, always-visible sequence (see render_stage_tracker in the widget
    cell below). Pregame stages run once per match; PLAY_*/WAIT_* alternate
    every turn depending on whose turn it is."""
    JOIN = 'join'
    RIDDLE_RECEIVED = 'riddle_received'
    RIDDLE_PROCESSED = 'riddle_processed'
    RIDDLE_PRINTED = 'riddle_printed'
    FREE_HINT_RECEIVED = 'free_hint_received'
    FREE_HINT_PROCESSED = 'free_hint_processed'
    FREE_HINT_READY = 'free_hint_ready'
    PLAY_FLIP = 'play_flip'
    PLAY_WAITING = 'play_waiting'
    PLAY_DETECTING = 'play_detecting'
    PLAY_COMPARING = 'play_comparing'
    PLAY_RESULT = 'play_result'
    WAIT_WAITING = 'wait_waiting'
    WAIT_DETECTING = 'wait_detecting'
    WAIT_LOGGING = 'wait_logging'
    GAME_OVER = 'game_over'


PREGAME_STAGES = [
    (Stage.JOIN, 'Join'),
    (Stage.RIDDLE_RECEIVED, 'Riddle Received'),
    (Stage.RIDDLE_PROCESSED, 'Riddle Processed'),
    (Stage.RIDDLE_PRINTED, 'Riddle Printed'),
    (Stage.FREE_HINT_RECEIVED, 'Hint Received'),
    (Stage.FREE_HINT_PROCESSED, 'Hint Processed'),
    (Stage.FREE_HINT_READY, 'Hint Ready'),
]
PLAY_STAGES = [
    (Stage.PLAY_FLIP, 'Flip'),
    (Stage.PLAY_WAITING, 'Waiting'),
    (Stage.PLAY_DETECTING, 'Detecting'),
    (Stage.PLAY_COMPARING, 'Comparing'),
    (Stage.PLAY_RESULT, 'Result'),
]
WAIT_STAGES = [
    (Stage.WAIT_WAITING, 'Waiting'),
    (Stage.WAIT_DETECTING, 'Detecting'),
    (Stage.WAIT_LOGGING, 'Logging'),
]


class MatchClient:
    POLL_INTERVAL_SECONDS = 0.3
    HINT_WAIT_TIMEOUT_SECONDS = 5.0
    # Matches the referee's PHYSICAL_FLIP_OFFSET (game_state.rs / data/game_config.json)
    # -- the fixed overhead the scoring tiers already assume for a physical flip +
    # camera re-capture. Tune to your actual physical setup's flip speed if it differs.
    PHYSICAL_FLIP_DELAY_SECONDS = 20

    def __init__(self, referee_client, on_update=None):
        self.client = referee_client
        self.on_update = on_update or (lambda: None)
        self.lock = threading.Lock()

        self._stop_event = threading.Event()
        self._pause_event = threading.Event()
        self._thread = None
        self._reveal_queue = queue.Queue()
        self._reveal_thread = None

        self.stage = Stage.JOIN
        self.play_mode = False       # Wait Mode by default -- won't act on its own turn until enabled
        self.teams = []
        self.total_pairs = 0
        self.match_number = 0        # local count of game_start messages received on this connection
        self.turn_number = 0         # local action-cycle count; server flip_num is stored separately
        self.current_turn = None     # active turn record shown in the UI
        self.turn_history = []       # completed/current turn records for review
        self.reveal_history = []     # every reveal, including opponent turns
        self.confirmed_matches = []  # authoritative match messages from the server
        self.selected_positions = []  # the two cards output for our current flip_both
        self.unavailable_positions = set()  # locally exclude positions rejected by the server
        self.server_truth_positions = set()  # positions corrected/confirmed by authoritative replies
        self.board_memory = {}       # pos -> class name, from every card_revealed seen
        self.matched_positions = set()
        self.pending = []            # this turn's own flips: [{'pos', 'cls', 'score'}, ...]
        self.awaiting_positions = set()  # our own outstanding flip request(s), to tell them apart from the opponent's
        self.pending_hint_object = None  # queued by queue_hint(), sent at the start of our next turn
        self.my_turn = False
        self.active_team = None
        self.scores = {}
        self.pairs_remaining = None
        self.last_hint = None
        self.last_revealed_pos = None      # set the instant a card_revealed arrives, cleared once detection runs
        self.last_hint_row_png_base64 = None  # from hint_response -- shown in the status panel either way
        self.last_hint_col_png_base64 = None
        self.last_hint_position = None   # auto-decoded from the above via MNIST, e.g. 'C5' -- None if decode failed
        self.last_hint_object = None     # which object the pending/last hint request was for
        self.pregame_riddle = None       # from pregame_riddle -- auto-solved via the LLM below
        self.pregame_riddle_answer = None  # LLM's best-guess answer, or None if solving failed
        self.free_hint_fragments = {}    # index -> fragment text, filled in as free_hint_fragment arrives
        self.free_hint_total = None
        self.free_hint_text = None       # assembled once every fragment 0..total-1 has arrived
        self.game_over = False
        self.winner = None
        self.robot_id = None          # from game_start: which Genesis arm is ours (0=red, 1=blue)
        self.genesis_team_id = None   # from game_start: "team_red"/"team_blue" for pynqsim.join_competition()
        self.genesis_url = None       # from game_start: pynqsim server base URL
        self.genesis_sim = None       # live pynqsim.SimulationClient once connected this match, else None
        self.log = []                # raw {'direction', 'message'} entries for the transparency panel

    # -- lifecycle -----------------------------------------------------
    def start(self):
        if self._thread is not None:
            raise RuntimeError('Match client already started.')
        self._stop_event.clear()
        self._pause_event.clear()
        self._reveal_queue = queue.Queue()
        self._reveal_thread = threading.Thread(target=self._run_reveal_worker, daemon=True)
        self._reveal_thread.start()
        self._thread = threading.Thread(target=self._run_loop, daemon=True)
        self._thread.start()

    def pause(self):
        if self.my_turn and self.pending:
            print('[match] WARNING: pausing mid-turn does not pause the referee\'s 120s turn timer.')
        self._pause_event.set()

    def resume(self):
        self._pause_event.clear()

    def stop(self):
        self._stop_event.set()
        self._reveal_queue.put(None)
        if self._thread is not None:
            self._thread.join(timeout=2.0)
        if self._reveal_thread is not None:
            self._reveal_thread.join(timeout=2.0)
        self._thread = None
        self._reveal_thread = None

    # -- externally-triggered actions -----------------------------------
    def set_play_mode(self, enabled):
        """Toggle autonomous play. Turning it on while it's already our turn (and we
        haven't flipped anything yet) kicks off this turn immediately in a background
        thread, rather than waiting for a future your_turn that may not come for a while."""
        with self.lock:
            self.play_mode = enabled
            should_start_now = enabled and self.my_turn and not self.pending and not self.awaiting_positions
        if should_start_now:
            threading.Thread(target=self._start_turn, daemon=True).start()

    def queue_hint(self, obj):
        """Send immediately. The referee queues requests received while we wait
        and resolves them before the next your_turn, outside our turn clock."""
        obj = str(obj).strip()
        if not obj:
            raise ValueError('Hint object cannot be empty.')
        with self.lock:
            self.pending_hint_object = obj
            self.last_hint_object = obj
        self._log('send', {'type': 'hint_request', 'team': self.client.team, 'object': obj})
        self.client.request_hint(obj)
        self.on_update()

    def _solve_pregame_riddle_background(self, riddle_text):
        """Solve off the poll thread so slow LLM I/O never stops referee messages."""
        try:
            result = solve_pregame_riddle_with_llm(riddle_text)
            with self.lock:
                if self.pregame_riddle != riddle_text or self._stop_event.is_set():
                    return
                self.pregame_riddle_answer = result['answer']
                self.stage = Stage.RIDDLE_PROCESSED
            self.on_update()
            fallback_note = ' (plain-text fallback)' if result.get('used_fallback') else ''
            print(f'>>> PRE-GAME RIDDLE ANSWER{fallback_note}: {result["answer"]} <<<')
            with self.lock:
                if self.pregame_riddle == riddle_text and not self._stop_event.is_set():
                    self.stage = Stage.RIDDLE_PRINTED
            self.on_update()
        except Exception as exc:
            print(f'[match] could not auto-solve the pre-game riddle: {exc}')

    def _begin_turn_record(self, active_team, mine, flip_num=None):
        """Record the server-announced active team and preserve the server's
        optional flip_num without inventing fields in any wire message."""
        with self.lock:
            self.turn_number += 1
            self.active_team = active_team
            self.current_turn = {
                'turn_number': self.turn_number,
                'active_team': active_team,
                'mine': bool(mine),
                'flip_num': flip_num,
                'selected': [],
                'reveals': [],
                'result': None,
            }
            self.turn_history.append(self.current_turn)

    def _finish_current_turn(self, result):
        with self.lock:
            if self.current_turn is not None:
                self.current_turn['result'] = dict(result)

    # -- autonomous turn logic --------------------------------------------
    def _start_turn(self):
        if not self.play_mode:
            return
        with self.lock:
            self.stage = Stage.PLAY_FLIP
        self.on_update()
        self._auto_flip_turn()

    def _auto_flip_turn(self):
        try:
            pos1, pos2 = self._choose_pair()
            self._flip_both(pos1, pos2)
        except RuntimeError as exc:
            print(f'[match] auto-flip failed: {exc}')
        self.on_update()

    def _choose_pair(self):
        """Prefer a guaranteed known pair, then pair a known position with an
        unrevealed one, then two fresh unrevealed positions. If the board is
        fully revealed but nothing pairs up by our own detection (only
        possible after a misdetection -- a wrong guess here is caught by
        re-observing the position again, not by the referee correcting it),
        retry two already-revealed-but-unmatched positions.
        """
        with self.lock:
            board_memory = dict(self.board_memory)
            unavailable = set(self.matched_positions) | set(self.unavailable_positions)

        by_cls = {}
        for pos, cls in board_memory.items():
            if pos in unavailable or cls == 'unknown':
                continue
            by_cls.setdefault(cls, []).append(pos)
        for positions in by_cls.values():
            if len(positions) >= 2:
                return positions[0], positions[1]

        unrevealed = [
            pos_name(row, col)
            for row in range(GRID_ROWS) for col in range(GRID_COLS)
            if pos_name(row, col) not in board_memory and pos_name(row, col) not in unavailable
        ]
        known_pos = next(iter(values[0] for values in by_cls.values()), None)
        if known_pos is not None and unrevealed:
            return known_pos, unrevealed[0]
        if len(unrevealed) >= 2:
            return unrevealed[0], unrevealed[1]

        unmatched_revealed = [pos for pos in board_memory if pos not in unavailable]
        if len(unmatched_revealed) < 2:
            raise RuntimeError('Fewer than two available positions remain; waiting for game_over.')
        return unmatched_revealed[0], unmatched_revealed[1]

    def _flip_both(self, pos1, pos2):
        with self.lock:
            if not self.my_turn:
                raise RuntimeError('Not your turn.')
            if self.pending or self.awaiting_positions:
                raise RuntimeError('Already flipped this turn.')
            if pos1 == pos2:
                raise RuntimeError('Cannot flip the same position twice.')
            self.selected_positions = [pos1, pos2]
            self.awaiting_positions = {pos1, pos2}
            if self.current_turn is not None:
                self.current_turn['selected'] = [pos1, pos2]
        self._log('send', {'type': 'flip_both', 'team': self.client.team, 'pos1': pos1, 'pos2': pos2})
        self.client.flip_both(pos1, pos2)
        with self.lock:
            self.stage = Stage.PLAY_WAITING
        self.on_update()
        self._genesis_flip_card(pos1)
        self._genesis_flip_card(pos2)

    # -- Genesis (optional, cosmetic only) --------------------------------
    def _connect_genesis(self):
        """Best-effort: joins Genesis's competition-mode scene as our
        assigned team so _genesis_flip_card/_genesis_end_turn below can
        animate our arm. Any failure (pynqsim not installed, server
        unreachable, already joined) is logged and swallowed, never
        raised -- Genesis is purely cosmetic and never gets a vote in the
        real match, which is decided entirely by report_result."""
        self.genesis_sim = None
        if not (self.genesis_team_id and self.genesis_url):
            return
        try:
            from pynqsim import SimulationClient
            from urllib.parse import urlparse
            parsed = urlparse(self.genesis_url)
            sim = SimulationClient(parsed.hostname, port=parsed.port or 9002)
            sim.join_competition(team_id=self.genesis_team_id)
            self.genesis_sim = sim
            print(f'[genesis] joined as {self.genesis_team_id} at {self.genesis_url}')
        except Exception as exc:
            print(f'[genesis] connection skipped (cosmetic only, match unaffected): {exc}')
            self.genesis_sim = None

    def _genesis_flip_card(self, pos):
        """Mirrors a real flip onto the simulated arm, purely for visual
        effect -- the referee's own report_result claim is what actually
        decides the match either way."""
        if self.genesis_sim is None:
            return
        try:
            row, col = parse_pos(pos)
            self.genesis_sim.flip_card(row, col)
        except Exception as exc:
            print(f'[genesis] flip_card({pos}) failed (cosmetic only): {exc}')

    def _genesis_end_turn(self):
        if self.genesis_sim is None:
            return
        try:
            self.genesis_sim.end_turn()
        except Exception as exc:
            print(f'[genesis] end_turn failed (cosmetic only): {exc}')

    def _disconnect_genesis(self):
        if self.genesis_sim is None:
            return
        try:
            self.genesis_sim.leave_competition()
        except Exception as exc:
            print(f'[genesis] leave_competition failed (cosmetic only): {exc}')
        self.genesis_sim = None

    # -- background loop --------------------------------------------------
    def _run_loop(self):
        while not self._stop_event.is_set():
            if self._pause_event.is_set():
                time.sleep(self.POLL_INTERVAL_SECONDS)
                continue
            try:
                for message in self.client.poll():
                    self._handle_message(message)
            except Exception as exc:
                print(f'[match] poll error: {type(exc).__name__}: {exc}')
            time.sleep(self.POLL_INTERVAL_SECONDS)

    def _handle_message(self, message):
        self._log('recv', message)
        message_type = message.get('type')

        if message_type == 'game_start':
            self._disconnect_genesis()
            with self.lock:
                self.match_number += 1
                self.teams = list(message['teams'])
                self.total_pairs = int(message['total_pairs'])
                self.turn_number = 0
                self.current_turn = None
                self.turn_history = []
                self.reveal_history = []
                self.confirmed_matches = []
                self.selected_positions = []
                self.unavailable_positions = set()
                self.server_truth_positions = set()
                self.board_memory = {}
                self.matched_positions = set()
                self.pending = []
                self.awaiting_positions = set()
                self.pending_hint_object = None
                self.last_hint = None
                self.last_hint_row_png_base64 = None
                self.last_hint_col_png_base64 = None
                self.last_hint_position = None
                self.last_hint_object = None
                self.my_turn = False
                self.active_team = None
                self.scores = {team: 0 for team in self.teams}
                self.pairs_remaining = self.total_pairs
                self.game_over = False
                self.winner = None
                self.robot_id = message.get('robot_id')
                self.genesis_team_id = message.get('genesis_team_id')
                self.genesis_url = message.get('genesis_url')
            self._connect_genesis()
        elif message_type == 'your_turn':
            self._begin_turn_record(self.client.team, mine=True, flip_num=message.get('flip_num'))
            with self.lock:
                self.my_turn = True
                self.pending = []
                self.awaiting_positions = set()
                self.selected_positions = []
                if self.pending_hint_object:
                    self.last_hint = f'No response for {self.pending_hint_object!r} (server silently refused it).'
                    self.pending_hint_object = None
                self.stage = Stage.PLAY_FLIP
            self.on_update()
            self._start_turn()
            return
        elif message_type == 'wait':
            self._begin_turn_record(message['active_team'], mine=False)
            with self.lock:
                self.my_turn = False
                self.selected_positions = []
                self.stage = Stage.WAIT_WAITING
            self._genesis_end_turn()
        elif message_type == 'card_revealed':
            self._queue_card_revealed(message['pos'])
        elif message_type == 'invalid':
            with self.lock:
                rejected_positions = list(self.selected_positions)
                self.unavailable_positions.update(rejected_positions)
                self.awaiting_positions = set()
                self.pending = []
                self.selected_positions = []
                self.stage = Stage.PLAY_FLIP
                if self.current_turn is not None:
                    self.current_turn.setdefault('invalid_attempts', []).append({
                        'positions': rejected_positions,
                        'reason': message['reason'],
                    })
                should_retry = self.play_mode and self.my_turn and not self.game_over
            print(f"[match] flip rejected: {message['reason']}")
            if should_retry:
                threading.Thread(target=self._auto_flip_turn, daemon=True).start()
        elif message_type == 'match':
            with self.lock:
                self.matched_positions.update([message['pos1'], message['pos2']])
                self.unavailable_positions.update([message['pos1'], message['pos2']])
                self.server_truth_positions.update([message['pos1'], message['pos2']])
                if message.get('cls'):
                    self.board_memory[message['pos1']] = message['cls']
                    self.board_memory[message['pos2']] = message['cls']
                self.scores = dict(message['scores'])
                self.pairs_remaining = message['remaining']
                self.pending = []
                confirmed = {
                    'match_number': self.match_number,
                    'turn_number': self.turn_number,
                    'pos1': message['pos1'],
                    'pos2': message['pos2'],
                    'class': message.get('cls'),
                    'scorer': message.get('scorer'),
                }
                self.confirmed_matches.append(confirmed)
                if self.current_turn is not None:
                    self.current_turn['result'] = {'type': 'match', **confirmed}
        elif message_type == 'no_match':
            with self.lock:
                # The referee deliberately doesn't send the real classes at
                # pos1/pos2 here -- a misdetected pair has to be caught by
                # re-observing it yourself, not by the referee handing you
                # the answer on a wrong guess. Scoring is unaffected either
                # way: it's always decided server-side against the grid's
                # real answer key, never from what a client reports.
                self.scores = dict(message['scores'])
                self.pending = []
                if self.current_turn is not None:
                    self.current_turn['result'] = {
                        'type': 'no_match',
                        'pos1': message['pos1'], 'pos2': message['pos2'],
                    }
        elif message_type == 'pregame_riddle':
            # Sent to both teams during the pre-game window. Auto-solved via
            # the halo Strix LLM below -- tell the human referee out loud if
            # you're first to answer correctly. There is no wire message for
            # submitting an answer, judging is entirely manual.
            with self.lock:
                self.pregame_riddle = message['riddle']
                self.pregame_riddle_answer = None
                self.free_hint_fragments = {}
                self.free_hint_total = None
                self.free_hint_text = None
                self.stage = Stage.RIDDLE_RECEIVED
            self.on_update()
            threading.Thread(
                target=self._solve_pregame_riddle_background,
                args=(self.pregame_riddle,),
                daemon=True,
                name='pregame-riddle-solver',
            ).start()
        elif message_type == 'free_hint_fragment':
            # One shared, non-competitive hint, split into plain-text
            # fragments -- assemble every index 0..total-1 yourself.
            with self.lock:
                self.free_hint_fragments[message['index']] = message['text']
                self.free_hint_total = message['total']
                self.stage = Stage.FREE_HINT_RECEIVED
                if len(self.free_hint_fragments) == self.free_hint_total:
                    ordered = [self.free_hint_fragments[i] for i in range(self.free_hint_total)]
                    self.free_hint_text = ' '.join(ordered)
                    self.stage = Stage.FREE_HINT_PROCESSED
            if self.stage == Stage.FREE_HINT_PROCESSED:
                self.on_update()
                with self.lock:
                    self.stage = Stage.FREE_HINT_READY
        elif message_type == 'hint_response':
            # Row and column arrive as small digit photos, not text --
            # decode them ourselves via the on-board MNIST classifier
            # (same model used by the Paid Hint debug panel) rather than
            # relying on a human to read them off the screen.
            self.pending_hint_object = None
            self.last_hint_row_png_base64 = message['row_digit_png_base64']
            self.last_hint_col_png_base64 = message['col_digit_png_base64']
            try:
                row_digit = decode_digit_png_base64(self.last_hint_row_png_base64)
                col_digit = decode_digit_png_base64(self.last_hint_col_png_base64)
                # Digits are 1-indexed (row A=1, col 1=1) to match position
                # naming directly -- pos_name() itself is 0-indexed.
                self.last_hint_position = pos_name(row_digit - 1, col_digit - 1)
                self.last_hint = f'Paid hint resolved: row={row_digit}, col={col_digit} -> {self.last_hint_position}'
                if self.last_hint_object:
                    # Feed straight into board_memory -- _choose_pair()'s
                    # existing tiered logic (guaranteed match / hedge /
                    # explore) picks this up automatically on the very
                    # next call, no special-casing needed there. Unlike a
                    # no_match reveal, this is the team's own deliberate
                    # paid action, not the referee correcting a guess.
                    self.board_memory[self.last_hint_position] = self.last_hint_object
            except Exception as exc:
                self.last_hint_position = None
                self.last_hint = f'row/col digit images received, but MNIST decode failed ({exc}) -- see images below'
        elif message_type == 'hint_rejected':
            self.pending_hint_object = None
            self.last_hint = f"rejected: {message['reason']}"
        elif message_type == 'game_over':
            with self.lock:
                self.game_over = True
                self.winner = message['winner']
                self.scores = message['scores']
                self.stage = Stage.GAME_OVER
            self._disconnect_genesis()

        self.on_update()

    def _queue_card_revealed(self, pos):
        """Capture ownership immediately, then let the poll loop keep draining
        while one serialized worker handles physical delay and camera access."""
        with self.lock:
            is_mine = pos in self.awaiting_positions
            if is_mine:
                self.awaiting_positions.discard(pos)
            turn_record = self.current_turn
            item = {
                'pos': pos,
                'mine': is_mine,
                'match_number': self.match_number,
                'turn_number': self.turn_number,
                'active_team': self.active_team,
                'turn_record': turn_record,
            }
            self.stage = Stage.PLAY_DETECTING if is_mine else Stage.WAIT_DETECTING
            self.last_revealed_pos = pos
        print(f'>>> PHYSICAL REFEREE: flip the real card at position {pos} now <<<')
        self.on_update()
        self._reveal_queue.put(item)

    def _run_reveal_worker(self):
        while not self._stop_event.is_set():
            item = self._reveal_queue.get()
            if item is None:
                self._reveal_queue.task_done()
                break
            try:
                while self._pause_event.is_set() and not self._stop_event.is_set():
                    time.sleep(self.POLL_INTERVAL_SECONDS)
                if not self._stop_event.is_set():
                    self._process_card_revealed(item)
            except Exception as exc:
                print(f"[match] reveal processing failed for {item['pos']}: {type(exc).__name__}: {exc}")
            finally:
                self._reveal_queue.task_done()

    def _process_card_revealed(self, item):
        pos = item['pos']
        is_mine = item['mine']
        with self.lock:
            if self.current_turn is item['turn_record'] and not self.game_over:
                self.stage = Stage.PLAY_DETECTING if is_mine else Stage.WAIT_DETECTING
        self.on_update()
        time.sleep(self.PHYSICAL_FLIP_DELAY_SECONDS)
        with self.lock:
            if item['match_number'] != self.match_number:
                return  # stale queued reveal from a previous scheduled match
            if self.last_revealed_pos == pos:
                self.last_revealed_pos = None
        try:
            cls, score = detect_position(pos, show=True)
        except Exception as exc:
            print(f'[detect] {pos} failed: {type(exc).__name__}: {exc}')
            cls, score = 'unknown', 0.0

        should_submit = False
        with self.lock:
            if pos not in self.server_truth_positions:
                self.board_memory[pos] = cls
            reveal = {
                'match_number': item['match_number'],
                'turn_number': item['turn_number'],
                'active_team': item['active_team'],
                'mine': is_mine,
                'pos': pos,
                'class': cls,
                'score': float(score),
                'timestamp': time.time(),
            }
            self.reveal_history.append(reveal)
            turn_record = item['turn_record']
            if turn_record is not None:
                turn_record['reveals'].append(reveal)
            if is_mine:
                self.pending.append({'pos': pos, 'cls': cls, 'score': score})
                should_submit = len(self.pending) == 2 and not self.game_over
                if not should_submit and self.current_turn is turn_record and not self.game_over:
                    self.stage = Stage.PLAY_DETECTING
            elif self.current_turn is turn_record and not self.game_over:
                self.stage = Stage.WAIT_LOGGING

        self.on_update()
        if is_mine and should_submit:
            self._submit_result()
        elif not is_mine and self.current_turn is item['turn_record'] and not self.game_over:
            with self.lock:
                if not self.game_over:
                    self.stage = Stage.WAIT_WAITING
            self.on_update()

    def _submit_result(self):
        with self.lock:
            self.stage = Stage.PLAY_COMPARING
            first, second = self.pending[0], self.pending[1]
        self.on_update()
        # Two failed detections must never become a false "unknown == unknown" match.
        claim = 'match' if first['cls'] != 'unknown' and first['cls'] == second['cls'] else 'no_match'
        self._log('send', {
            'type': 'report_result', 'team': self.client.team,
            'pos1': first['pos'], 'pos2': second['pos'],
            'cls1': first['cls'], 'cls2': second['cls'], 'claim': claim,
        })
        self.client.report_result(first['pos'], second['pos'], first['cls'], second['cls'], claim)
        with self.lock:
            self.stage = Stage.PLAY_RESULT

    def _log(self, direction, message):
        with self.lock:
            self.log.append({'direction': direction, 'message': message})
            self.log = self.log[-200:]


## 10. Widget GUI

Connection fields, Start/Pause/Resume/Stop, a Wait Mode / Play Mode safety toggle, a hint queue, and a
live status + raw wire-message log so you can see exactly what's being sent and received if you get
stuck. There is no per-turn control -- once you're in Play Mode, the match plays itself.


In [ ]:
server_text = widgets.Text(value=SERVER, description='Server:')
key_text = widgets.Text(value=BROKER_KEY, description='Key:')
referee_text = widgets.Text(value=REFEREE_ID, description='Referee ID:')
master_text = widgets.Text(value=MASTER_ID, description='Master ID:')
team_text = widgets.Text(value=TEAM_NAME, description='Team:')
team_secret_text = widgets.Text(value=TEAM_SECRET, description='Team secret:',
                                 placeholder='leave blank to skip self-reporting your MAC')
board_id_text = widgets.Text(value=BOARD_ID_OVERRIDE, description='Board ID override:',
                              placeholder='leave blank to use pynqp2p.get_id()')

test_connectivity_button = widgets.Button(description='Test Connectivity', button_style='info')
connect_button = widgets.Button(description='Connect', button_style='primary')
disconnect_button = widgets.Button(description='Disconnect', button_style='danger', disabled=True)
start_button = widgets.Button(description='Start', button_style='success', disabled=True)
pause_button = widgets.Button(description='Pause', disabled=True)
resume_button = widgets.Button(description='Resume', disabled=True)
stop_button = widgets.Button(description='Stop', button_style='danger', disabled=True)

control_mode_toggle = widgets.ToggleButtons(
    options=[('Manual', 'manual'), ('Auto', 'auto')],
    value='manual', description='Control:',
)
play_mode_toggle = widgets.ToggleButtons(
    options=[('Wait Mode', False), ('Play Mode', True)],
    value=False, description='Mode:', disabled=True,
)

hint_object_text = widgets.Text(value='', placeholder='e.g. dog', description='Hint object:')
hint_button = widgets.Button(description='Queue Hint', button_style='warning', disabled=True)

solve_riddle_button = widgets.Button(description='Solve Riddle (LLM)', button_style='info')

camera_debug_button = widgets.Button(description='Camera Debug One-Shot', button_style='info')
camera_debug_position_text = widgets.Text(value='A1', description='Grid cell:', layout=widgets.Layout(width='180px'))
camera_debug_position_button = widgets.Button(description='Debug Grid Cell', button_style='info')
detection_approach_dropdown = widgets.Dropdown(
    options=[
        ('YOLO - full frame', 'yolo_full_frame'),
        ('YOLO - grid crops', 'yolo_grid_crops'),
        ('ArUco - border markers', 'aruco_border'),
        ('ArUco - per-card markers', 'aruco_per_card'),
        ('ArUco border + grid crops', 'aruco_border_grid_crops'),
    ],
    value=DETECTION_APPROACH, description='Detection:',
)
camera_debug_output = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})

status_html = widgets.HTML(value='<i>Not connected.</i>')
board_html = widgets.HTML(value='')
show_raw_log_checkbox = widgets.Checkbox(value=False, description='Show raw wire messages', indent=False)
log_output = widgets.Output(layout={'border': '1px solid #ddd', 'height': '160px', 'overflow_y': 'auto'})
action_output = widgets.Output()

match = None  # the active MatchClient, set on Connect


def render_status():
    if match is None:
        status_html.value = '<i>Not connected.</i>'
        board_html.value = ''
        return

    flip_banner = ''
    if match.last_revealed_pos:
        flip_banner = (
            '<div style="background:#ffcc00;color:#000;padding:10px;font-size:20px;'
            'font-weight:bold;text-align:center;margin-bottom:8px">'
            f'FLIP THE PHYSICAL CARD AT {match.last_revealed_pos} NOW</div>'
        )

    mode_label = 'Play Mode' if match.play_mode else 'Wait Mode (not auto-playing)'
    if match.game_over:
        banner = f"<h3>Game over -- winner: {match.winner}</h3>"
    elif match.my_turn:
        waiting_note = ' (waiting for reveal...)' if match.awaiting_positions else ''
        banner = f"<b>Your turn</b> ({mode_label}). Flips this turn: {len(match.pending)}/2{waiting_note}"
    else:
        banner = f"Waiting -- active team: {match.active_team} ({mode_label})"

    scores_text = ', '.join(f'{team}: {score}' for team, score in match.scores.items()) or 'no score yet'
    pairs_text = (
        f"{match.total_pairs - match.pairs_remaining}/{match.total_pairs} pairs"
        if match.pairs_remaining is not None else f"0/{match.total_pairs} pairs"
    )
    hint_text = f'<br>Last hint: {match.last_hint}' if match.last_hint else ''
    hint_images = ''
    if match.last_hint_row_png_base64 and match.last_hint_col_png_base64:
        hint_images = (
            '<br>Row: <img style="vertical-align:middle;background:#fff" '
            f'src="data:image/png;base64,{match.last_hint_row_png_base64}">'
            '&nbsp;&nbsp;Col: <img style="vertical-align:middle;background:#fff" '
            f'src="data:image/png;base64,{match.last_hint_col_png_base64}">'
        )
    pregame_text = ''
    if match.pregame_riddle:
        pregame_text = f'<br>Pre-game riddle: {match.pregame_riddle}'
        if match.pregame_riddle_answer:
            pregame_text += (
                '<br><span style="background:#2ea043;color:#fff;padding:2px 8px;'
                f'border-radius:4px;font-weight:bold">LLM answer: {match.pregame_riddle_answer}</span>'
            )
    if match.free_hint_text:
        free_hint_text = f'<br>Free hint (assembled): {match.free_hint_text}'
    elif match.free_hint_fragments:
        free_hint_text = f'<br>Free hint: {len(match.free_hint_fragments)}/{match.free_hint_total} fragments received'
    else:
        free_hint_text = ''
    genesis_text = ''
    if match.genesis_team_id:
        connection_state = 'connected' if match.genesis_sim is not None else 'not connected'
        genesis_text = (
            '<br><span style="background:#0d419d;color:#fff;padding:2px 8px;border-radius:4px">'
            f'&#x1F916; Genesis: {match.genesis_team_id} ({connection_state}) &nbsp; {match.genesis_url}</span>'
        )
    status_html.value = (
        f'{flip_banner}{banner}<br>Scores: {scores_text} &nbsp; ({pairs_text})'
        f'{hint_text}{hint_images}{pregame_text}{free_hint_text}{genesis_text}'
    )

    cells = []
    for row in range(GRID_ROWS):
        row_cells = []
        for col in range(GRID_COLS):
            pos = pos_name(row, col)
            label = match.board_memory.get(pos, '&nbsp;')
            color = '#d9f2d9' if pos in match.board_memory else '#f5f5f5'
            row_cells.append(f'<td style="border:1px solid #bbb;padding:6px;background:{color};min-width:70px;text-align:center">{pos}<br><small>{label}</small></td>')
        cells.append('<tr>' + ''.join(row_cells) + '</tr>')
    board_html.value = '<table style="border-collapse:collapse">' + ''.join(cells) + '</table>'

    start_button.disabled = True
    play_mode_toggle.disabled = match.game_over
    hint_button.disabled = match.game_over
    pause_button.disabled = match.game_over
    resume_button.disabled = match.game_over
    stop_button.disabled = False

    if show_raw_log_checkbox.value:
        with log_output:
            clear_output(wait=True)
            for entry in match.log[-40:]:
                arrow = '->' if entry['direction'] == 'send' else '<-'
                print(f"{arrow} {json.dumps(entry['message'])}")


def on_test_connectivity_clicked(_button):
    """Quick, fast-failing check that this board can actually reach the broker
    before attempting Connect -- pynqp2p's own requests calls have no timeout,
    so a genuinely unreachable broker makes Connect hang forever instead of
    failing with a clear error. Run this first."""
    with action_output:
        clear_output(wait=True)
        server = server_text.value.strip()
        key = key_text.value.strip()
        if not server:
            print('Enter a Server address first.')
            return
        url = f'http://{server}/ping'
        print(f'Testing connectivity to {url} ...')
        start = time.time()
        try:
            response = requests.post(url, data={'key': key, 'id': 'connectivity-test'}, timeout=8)
            elapsed_ms = int((time.time() - start) * 1000)
            if response.status_code == 200:
                print(f'REACHABLE -- got {response.text!r} in {elapsed_ms}ms. Safe to Connect.')
            elif response.status_code == 401:
                print('Reached the broker, but the Key is wrong (401 Unauthorized). Check BROKER_KEY.')
            else:
                print(f'Reached the broker, but got an unexpected response: {response.status_code} {response.text!r}')
        except requests.exceptions.ConnectTimeout:
            elapsed_ms = int((time.time() - start) * 1000)
            print(
                f'NOT REACHABLE -- connection timed out after {elapsed_ms}ms. '
                'This board cannot reach the broker at all -- check network routing '
                '(relay/tunnel setup) before trying Connect, which will hang the same way.'
            )
        except requests.exceptions.ConnectionError as exc:
            print(f'NOT REACHABLE -- connection refused or DNS failure: {exc}')
        except Exception as exc:
            print(f'Connectivity test failed: {type(exc).__name__}: {exc}')


def on_connect_clicked(_button):
    global match
    with action_output:
        clear_output(wait=True)
        try:
            client = RefereeClient(
                server_text.value, key_text.value, referee_text.value, team_text.value,
                master_id=master_text.value or None,
                board_id=board_id_text.value or None,
            )
            if team_secret_text.value:
                client.join_competition(team_secret_text.value)
                print(f'Sent join_competition for team {client.team!r} (mac {client.board_id}).')
            match = MatchClient(client, on_update=render_status)
            # Start receiving messages immediately, not on the separate Start
            # click below -- join_competition() already satisfies the
            # referee's "team joined" gate at Connect time, so the pregame
            # riddle/free hint can arrive before Start is ever clicked. Every
            # autonomous action is separately gated by play_mode (off by
            # default), so it's safe to listen from Connect onward.
            match.start()
            print(f'Connected as board {client.board_id}, team {client.team!r}.')
            start_button.disabled = False
            connect_button.disabled = True
            disconnect_button.disabled = False
        except Exception:
            import traceback
            traceback.print_exc()


def on_disconnect_clicked(_button):
    global match
    with action_output:
        clear_output(wait=True)
        try:
            if match is not None:
                match.stop()
            match = None
            connect_button.disabled = False
            disconnect_button.disabled = True
            start_button.disabled = True
            play_mode_toggle.disabled = True
            hint_button.disabled = True
            pause_button.disabled = True
            resume_button.disabled = True
            stop_button.disabled = True
            render_status()
            print('Disconnected -- edit the connection fields above if needed, then click Connect again.')
        except Exception:
            import traceback
            traceback.print_exc()


def on_start_clicked(_button):
    with action_output:
        clear_output(wait=True)
        try:
            # match.start() already ran at Connect (see on_connect_clicked) --
            # this button now only arms the play controls below, it doesn't
            # start message reception.
            play_mode_toggle.disabled = False
            hint_button.disabled = False
            pause_button.disabled = False
            stop_button.disabled = False
            if control_mode_toggle.value == 'auto':
                play_mode_toggle.value = True  # triggers on_play_mode_changed -> match.set_play_mode(True)
                print('Match client started in Auto mode -- will play automatically as soon as game_start arrives.')
            else:
                print('Match client started in Wait Mode -- waiting for game_start. Switch to Play Mode when ready.')
        except Exception:
            import traceback
            traceback.print_exc()


def on_pause_clicked(_button):
    match.pause()
    pause_button.disabled = True
    resume_button.disabled = False


def on_resume_clicked(_button):
    match.resume()
    pause_button.disabled = False
    resume_button.disabled = True


def on_stop_clicked(_button):
    match.stop()
    start_button.disabled = False
    play_mode_toggle.disabled = True
    hint_button.disabled = True
    pause_button.disabled = True
    resume_button.disabled = True
    stop_button.disabled = True


def on_play_mode_changed(change):
    if match is not None:
        match.set_play_mode(change['new'])


play_mode_toggle.observe(on_play_mode_changed, names='value')


def on_hint_clicked(_button):
    obj = hint_object_text.value.strip()
    with action_output:
        clear_output(wait=True)
        if not obj:
            print('Enter an object name first.')
            return
        match.queue_hint(obj)
        print(f'Hint queued: {obj!r} -- will be requested at the start of your next turn (if not already begun).')


def on_solve_riddle_clicked(_button):
    with action_output:
        clear_output(wait=True)
        if match is None or not match.pregame_riddle:
            print('No pre-game riddle received yet.')
            return
        try:
            result = solve_pregame_riddle_with_llm(match.pregame_riddle)
            match.pregame_riddle_answer = result['answer']
            print(f"LLM answer: {result['answer']}")
            render_status()
        except Exception:
            import traceback
            traceback.print_exc()


def on_camera_debug_clicked(_button):
    with camera_debug_output:
        clear_output(wait=True)
        try:
            camera_debug_one_shot()
        except Exception:
            import traceback
            traceback.print_exc()


def on_camera_debug_position_clicked(_button):
    pos = camera_debug_position_text.value.strip().upper()
    with camera_debug_output:
        clear_output(wait=True)
        try:
            detect_position_debug(pos)
        except Exception:
            import traceback
            traceback.print_exc()


solve_riddle_button.on_click(on_solve_riddle_clicked)
camera_debug_button.on_click(on_camera_debug_clicked)
camera_debug_position_button.on_click(on_camera_debug_position_clicked)
detection_approach_dropdown.observe(lambda change: set_detection_approach(change['new']), names='value')
test_connectivity_button.on_click(on_test_connectivity_clicked)
connect_button.on_click(on_connect_clicked)
disconnect_button.on_click(on_disconnect_clicked)
start_button.on_click(on_start_clicked)
pause_button.on_click(on_pause_clicked)
resume_button.on_click(on_resume_clicked)
stop_button.on_click(on_stop_clicked)
hint_button.on_click(on_hint_clicked)
show_raw_log_checkbox.observe(lambda _change: render_status(), names='value')

controls = widgets.VBox([
    widgets.HTML('<h3>Connection</h3>'),
    widgets.HBox([server_text, key_text, referee_text, master_text]),
    widgets.HBox([team_text, team_secret_text]),
    widgets.HBox([test_connectivity_button, board_id_text, connect_button, disconnect_button]),
    widgets.HTML('<h3>Camera Test</h3>'),
    widgets.HTML('<small>Debug only: verify the camera and YOLO model are working before a match. Draws every detected object/box on the live frame. Does not touch match state or send anything to the referee.</small>'),
    detection_approach_dropdown,
    widgets.HBox([camera_debug_button]),
    widgets.HBox([camera_debug_position_text, camera_debug_position_button]),
    camera_debug_output,
    widgets.HTML('<h3>Match Control</h3>'),
    widgets.HBox([start_button, pause_button, resume_button, stop_button]),
    widgets.HTML('<small>Control: Manual uses the Wait/Play toggle below. Auto switches to Play Mode automatically as soon as Start is clicked -- no manual toggle needed.</small>'),
    control_mode_toggle,
    play_mode_toggle,
    widgets.HTML('<h3>Status</h3>'),
    status_html,
    board_html,
    widgets.HTML('<h3>Pre-Game Riddle</h3>'),
    widgets.HTML('<small>Auto-solved via the LLM as soon as it arrives -- this button re-solves manually if needed.</small>'),
    widgets.HBox([solve_riddle_button]),
    widgets.HTML('<h3>Hint</h3>'),
    widgets.HBox([hint_object_text, hint_button]),
    widgets.HTML('<h3>Log</h3>'),
    show_raw_log_checkbox,
    log_output,
    action_output,
])

render_status()
display(controls)


## 11. MNIST Paid-Hint Debug (on-board digit capture)

Ported from `PYNQ 301 - Memory Game Grid Detection` so the same on-board digit
capture is available directly in the real match client, not just the local
practice notebook. Type a card name, then capture a row digit and a column
digit -- either from the MNIST camera or, for testing without a working MNIST
setup, from a pre-saved sample image in `mnist_digits_0-9/`. The captured
`(row, col)` is stored in `paid_hints` using the same `<row letter><col
number>` position naming as everywhere else in this notebook (e.g. digit 1,
digit 2 -> `B3`).

This section only stores hints locally for your own reference -- it does not
send anything over the wire. Wiring MNIST into the referee's own `hint_response`
digit images (so the board reads them automatically instead of a human) is a
future step, not done here.


In [ ]:
MNIST_MODEL_PATHS = [
    NOTEBOOK_DIR / 'dpu_mnist_classifier.xmodel',
    Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/PYNQ 201 - MNIST/dpu_mnist_classifier.xmodel'),
    Path('/home/root/jupyter_notebooks/pynq-dpu/dpu_mnist_classifier.xmodel'),
]
MNIST_DIGITS_DIR_CANDIDATES = [
    NOTEBOOK_DIR / 'mnist_digits_0-9',
    Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/mnist_digits_0-9'),
]
mnist_runner = None
mnist_input_data = None
mnist_output_data = None
mnist_output_size = None
mnist_camera = None
mnist_camera_device = None
paid_hints = {}       # pos -> card name, e.g. {'B3': 'dog'}
paid_hint_state = {'active': False, 'name': '', 'row': None}


def mnist_model_path():
    for path in MNIST_MODEL_PATHS:
        if path.exists():
            return path
    raise FileNotFoundError('Could not find dpu_mnist_classifier.xmodel. Check the PYNQ 201 - MNIST notebook folder.')


def mnist_digits_dir():
    for path in MNIST_DIGITS_DIR_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError(
        'Could not find mnist_digits_0-9 folder. Copy it into bootcamp_sessions or this notebook folder.'
    )


def calculate_mnist_softmax(data):
    result = np.exp(data)
    return result / np.sum(result)


def ensure_mnist_runner():
    global mnist_runner, mnist_input_data, mnist_output_data, mnist_output_size
    if mnist_runner is not None:
        return mnist_runner

    overlay.load_model(str(mnist_model_path()))
    mnist_runner = overlay.runner
    input_tensors = mnist_runner.get_input_tensors()
    output_tensors = mnist_runner.get_output_tensors()
    shape_in = tuple(input_tensors[0].dims)
    shape_out = tuple(output_tensors[0].dims)
    mnist_output_size = int(output_tensors[0].get_data_size() / shape_in[0])
    mnist_input_data = [np.empty(shape_in, dtype=np.float32, order='C')]
    mnist_output_data = [np.empty(shape_out, dtype=np.float32, order='C')]
    return mnist_runner


def ensure_yolo_runner():
    """Reloads the YOLO model after an MNIST capture swapped the DPU over --
    call this before any further board detection."""
    global dpu, inputTensors, outputTensors, shapeIn, shapeOut0, shapeOut1, shapeOut2
    global input_data, output_data, image_buffer, mnist_runner
    overlay.load_model(str(MODEL_PATH))
    dpu = overlay.runner
    inputTensors = dpu.get_input_tensors()
    outputTensors = dpu.get_output_tensors()
    shapeIn = tuple(inputTensors[0].dims)
    shapeOut0 = tuple(outputTensors[0].dims)
    shapeOut1 = tuple(outputTensors[1].dims)
    shapeOut2 = tuple(outputTensors[2].dims)
    input_data = [np.empty(shapeIn, dtype=np.float32, order='C')]
    output_data = [
        np.empty(shapeOut0, dtype=np.float32, order='C'),
        np.empty(shapeOut1, dtype=np.float32, order='C'),
        np.empty(shapeOut2, dtype=np.float32, order='C'),
    ]
    image_buffer = input_data[0]
    mnist_runner = None
    return dpu


def preprocess_mnist_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    resized = cv2.resize(blurred, (28, 28), interpolation=cv2.INTER_AREA)

    # MNIST training data is white digit on black background. Invert bright paper backgrounds.
    if np.mean(resized) > 127:
        resized = 255 - resized

    normalized = np.asarray(resized / 255.0, dtype=np.float32)
    return np.expand_dims(normalized, axis=2)


def open_mnist_camera(device=None):
    global mnist_camera, mnist_camera_device
    if mnist_camera is not None and mnist_camera.isOpened():
        mnist_camera.release()

    if device is None:
        device = mnist_camera_device
    if device is None:
        candidates = [candidate for candidate in video_devices() if candidate != globals().get('camera_device')]
        device = candidates[0] if candidates else globals().get('camera_device')
    if device is None:
        raise RuntimeError('No MNIST camera device selected.')

    mnist_camera = open_camera(device)
    mnist_camera_device = device
    return mnist_camera


def read_mnist_camera_frame(device=None, read_attempts=8):
    if device is not None or mnist_camera is None or not mnist_camera.isOpened():
        open_mnist_camera(device)

    for _ in range(max(1, int(read_attempts))):
        ret, frame = mnist_camera.read()
        if ret and frame is not None:
            return frame
        mnist_camera.grab()
        ret, frame = mnist_camera.retrieve()
        if ret and frame is not None:
            return frame
        time.sleep(0.05)

    open_mnist_camera(device)
    ret, frame = mnist_camera.read()
    if ret and frame is not None:
        return frame
    raise RuntimeError('Could not read a frame from the MNIST camera.')


def run_mnist_model_from_processed(digit_image):
    runner = ensure_mnist_runner()
    mnist_input_data[0][0, ...] = digit_image.reshape(mnist_input_data[0].shape[1:])
    job_id = runner.execute_async(mnist_input_data, mnist_output_data)
    runner.wait(job_id)
    logits = mnist_output_data[0].reshape(1, mnist_output_size)[0]
    probabilities = calculate_mnist_softmax(logits)
    prediction = int(probabilities.argmax())
    return prediction, probabilities


def _show_mnist_prediction(frame, digit_image, prediction, confidence):
    annotated = draw_label(frame.copy(), f'MNIST prediction: {prediction} ({confidence:.2f})', frame.shape[1] // 2, 30, (0, 255, 0))
    _, encoded = cv2.imencode('.jpeg', annotated)
    mnist_display_handle.update(Image(data=encoded.tobytes()))


def predict_mnist_digit_from_frame(frame, show=True):
    digit_image = preprocess_mnist_frame(frame)
    prediction, probabilities = run_mnist_model_from_processed(digit_image)
    confidence = float(probabilities[prediction])

    if show:
        _show_mnist_prediction(frame, digit_image, prediction, confidence)

    # Restore YOLO so normal board detection still works after this capture.
    ensure_yolo_runner()
    print(f'MNIST prediction: {prediction} (confidence={confidence:.3f})')
    return prediction


def capture_mnist_digit(device=None, show=True):
    frame = read_mnist_camera_frame(device=device)
    return predict_mnist_digit_from_frame(frame, show=show)


def decode_digit_png_base64(png_base64, show=True):
    """Decodes a base64 PNG digit image (as delivered by hint_response)
    into a predicted digit 0-9 via the on-board MNIST classifier."""
    png_bytes = base64.b64decode(png_base64)
    frame = cv2.imdecode(np.frombuffer(png_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError('Could not decode the hint digit image.')
    return predict_mnist_digit_from_frame(frame, show=show)


def predict_mnist_digit_from_saved_image(digit_choice, show=True):
    """Debug-only: classify a pre-saved sample digit image from
    mnist_digits_0-9/ instead of the live MNIST camera -- lets you test the
    Paid Hint capture flow without a working MNIST camera or writing digits
    by hand."""
    image_path = mnist_digits_dir() / f'digit_{int(digit_choice)}.png'
    frame = cv2.imread(str(image_path))
    if frame is None:
        raise FileNotFoundError(f'Could not read saved digit image: {image_path}')
    return predict_mnist_digit_from_frame(frame, show=show)


def inject_paid_hint(card_name, row, col):
    card_name = str(card_name).strip()
    row = int(row)
    col = int(col)
    if not card_name:
        raise ValueError('Enter a card name before using Paid Hint.')
    if not (0 <= row < GRID_ROWS and 0 <= col < GRID_COLS):
        raise ValueError(f'Hint digits row={row} col={col} are outside the {GRID_ROWS}x{GRID_COLS} grid.')

    pos = pos_name(row, col)
    paid_hints[pos] = card_name
    print(f'Paid hint stored: {card_name} at {pos}')
    return {'pos': pos, 'description': card_name}


## 12. MNIST Paid-Hint Widget GUI


In [ ]:
_mnist_camera_options = video_devices() or ['(no camera found)']
mnist_camera_dropdown = widgets.Dropdown(
    options=_mnist_camera_options,
    value=next((d for d in _mnist_camera_options if d != camera_device), _mnist_camera_options[0]),
    description='MNIST cam:',
)
mnist_digit_source_toggle = widgets.ToggleButtons(
    options=[('Live Camera', 'camera'), ('Saved Image', 'saved')],
    value='camera',
    description='Digit source:',
)
saved_digit_dropdown = widgets.Dropdown(options=list(range(10)), value=0, description='Saved digit:')

mnist_display_handle = display(None, display_id=True)
mnist_output = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})

paid_hint_name_text = widgets.Text(value='', placeholder='card name, e.g. car', description='Hint name:')
paid_hint_button = widgets.Button(description='Paid Hint', button_style='warning')
paid_hint_capture_button = widgets.Button(description='Capture Row', button_style='warning', disabled=True)
paid_hint_capture_button.layout.display = 'none'
mnist_debug_button = widgets.Button(description='Debug MNIST One-Shot', button_style='primary')
paid_hints_show_button = widgets.Button(description='Show Paid Hints', button_style='info')
paid_hints_clear_button = widgets.Button(description='Clear Paid Hints', button_style='danger')


def _run_mnist_action(label, action):
    with mnist_output:
        clear_output(wait=True)
        print(label)
        try:
            action()
        except Exception:
            import traceback
            traceback.print_exc()


def _reset_paid_hint_capture_button():
    paid_hint_state['active'] = False
    paid_hint_state['name'] = ''
    paid_hint_state['row'] = None
    paid_hint_capture_button.description = 'Capture Row'
    paid_hint_capture_button.disabled = True
    paid_hint_capture_button.layout.display = 'none'


def on_paid_hint_clicked(_button):
    def action():
        card_name = paid_hint_name_text.value.strip()
        if not card_name:
            raise ValueError('Enter a card name before clicking Paid Hint.')
        paid_hint_state['active'] = True
        paid_hint_state['name'] = card_name
        paid_hint_state['row'] = None
        paid_hint_capture_button.description = 'Capture Row'
        paid_hint_capture_button.disabled = False
        paid_hint_capture_button.layout.display = ''
        print(f"Paid hint started for '{card_name}'. Capture the row digit, then the column digit.")
    _run_mnist_action('Paid Hint', action)


def on_paid_hint_capture_clicked(_button):
    def action():
        if not paid_hint_state['active']:
            raise RuntimeError('Click Paid Hint first.')

        if mnist_digit_source_toggle.value == 'saved':
            digit = predict_mnist_digit_from_saved_image(saved_digit_dropdown.value, show=True)
        else:
            digit = capture_mnist_digit(device=mnist_camera_dropdown.value, show=True)

        if paid_hint_state['row'] is None:
            paid_hint_state['row'] = digit
            paid_hint_capture_button.description = 'Capture Column'
            print(f'Captured row: {digit}. Now capture the column digit.')
            return

        row = paid_hint_state['row']
        col = digit
        result = inject_paid_hint(paid_hint_state['name'], row, col)
        _reset_paid_hint_capture_button()
        print(result)
    _run_mnist_action('Capture Digit', action)


def on_mnist_debug_clicked(_button):
    def action():
        capture_mnist_digit(device=mnist_camera_dropdown.value, show=True)
    _run_mnist_action('Debug MNIST One-Shot', action)


def on_paid_hints_show_clicked(_button):
    def action():
        if not paid_hints:
            print('No paid hints stored yet.')
        for pos, description in sorted(paid_hints.items()):
            print(f'  {pos}: {description}')
    _run_mnist_action('Paid Hints', action)


def on_paid_hints_clear_clicked(_button):
    def action():
        paid_hints.clear()
        print('Paid hints cleared.')
    _run_mnist_action('Clear Paid Hints', action)


paid_hint_button.on_click(on_paid_hint_clicked)
paid_hint_capture_button.on_click(on_paid_hint_capture_clicked)
mnist_debug_button.on_click(on_mnist_debug_clicked)
paid_hints_show_button.on_click(on_paid_hints_show_clicked)
paid_hints_clear_button.on_click(on_paid_hints_clear_clicked)

mnist_controls = widgets.VBox([
    widgets.HTML('<h3>MNIST Paid-Hint Debug</h3>'),
    widgets.HBox([mnist_camera_dropdown, mnist_digit_source_toggle, saved_digit_dropdown]),
    widgets.HBox([mnist_debug_button]),
    widgets.HTML('<small>Type a card name, click Paid Hint, then capture the row digit and the column digit. Switch to Saved Image to test with mnist_digits_0-9/ instead of a live camera.</small>'),
    widgets.HBox([paid_hint_name_text, paid_hint_button, paid_hint_capture_button]),
    widgets.HBox([paid_hints_show_button, paid_hints_clear_button]),
    mnist_output,
])

display(mnist_controls)


## 13. Cleanup

Run at the end of a match/session.


In [ ]:
if match is not None:
    match.stop()
cap.release()
